In [ ]:
"""
================================================================================
 VQC (Variational Quantum Circuit) Training and Testing Pipeline
 for Multiclass Classification via One-vs-Rest (sklearn)
================================================================================

This script:
  1. Supports loading data with two configurable embedding types:
        - "amplitude" -> Amplitude Embedding (qml.AmplitudeEmbedding)
        - "angle"     -> Angle/Phase Embedding (qml.AngleEmbedding, RY rotations)
  2. Implements five ansätze (variational circuits) known in the literature,
     each defined as a SINGLE-layer block (rather than the complete stack),
     enabling the "data re-uploading" scheme: the input data are re-uploaded
     at each of the n_layers layers — layer =
     [data loading + ansatz block] — which increases circuit expressivity
     (Pérez-Salinas et al., Quantum 2020):
        - hardware_efficient : Hardware-Efficient Ansatz (Kandala et al., 2017)
        - strongly_entangling: Strongly Entangling Layers (Schuld et al. / PennyLane)
        - basic_entangler    : Basic Entangler Layers (simplified HEA variant)
        - qaoa_inspired      : QAOA-inspired ansatz (Farhi & Neven, 2018)
        - tree_tensor        : Tree Tensor Network Ansatz (Grant et al., 2018)
  3. Creates a binary VQC classifier compatible with sklearn (BaseEstimator +
     ClassifierMixin), which is then wrapped by sklearn's OneVsRestClassifier
     to create a multiclass classifier.
  4. Efficiently trains circuit parameters using JAX (autodiff + jit) and
     Optax (Adam optimizer), with the QNode running in "backprop" mode
     (exact gradients through automatic differentiation, much faster than
     parameter-shift for classical simulation) and vectorized over the batch
     with jax.vmap.
  5. Supports dimensionality reduction through PCA or UMAP, with the final
     number of features (= number of qubits for the "angle" embedding)
     configurable by the user. PCA/UMAP and normalization are fitted only on
     training data: MinMax [0,pi] in the quantum branch and MinMax [0,1] in
     the classical branch inside the GridSearchCV Pipeline, preventing data leakage.
  6. Trains and evaluates the five ansätze on the same dataset, saving
     accuracy, precision, recall, f1-score, and the trained parameters of each
     circuit in a dictionary serialized as a pickle file.
  7. Optionally trains/evaluates classical baselines — SVM
     (sklearn.svm.SVC, linear/rbf/sigmoid/poly kernels), Decision Tree
     (sklearn.tree.DecisionTreeClassifier), MLP/neural network
     (sklearn.neural_network.MLPClassifier, with an architecture grid +
     early stopping), and Random Forest (sklearn.ensemble.RandomForestClassifier)
     — each through GridSearchCV on the SAME training and test sets used by
     the VQCs, storing their results in the same dictionary/plot/pickle for
     direct comparison.
  8. Includes a dedicated pipeline (run_mental_health_pipeline) for a
     mental-health dataset in which main disorders are subdivided into
     specific disorders: for EACH main disorder, it trains a multiclass
     classifier whose classes are the specific disorders associated with
     that disorder plus the healthy control group, and saves a pickle
     containing results for all disorders.
  9. Supports repeating the entire experiment N_REPEAT times (parameter
     n_repeats), with each repetition using a different train/test split
     (different random_state). In EACH repetition, the SAME split is used
     to train both the VQCs and ALL classical baselines, ensuring a fair
     comparison; results are aggregated as mean ± standard deviation per
     model (see train_evaluate_all_models / aggregate_repetitions).
  10. Includes generate_comparison_table, which produces a table (DataFrame)
      containing the best configuration for each MODEL TYPE — VQC, SVM,
      Decision Tree, MLP, and Random Forest (according to the selected metric)
      — for each dataset. It works both with the result of run_pipeline
      (one dataset) and run_mental_health_pipeline (one row per model type
      and main disorder).
  11. Saved results are organized by CONFIGURATION KEY (see
      generate_configuration_key): rerunning run_pipeline or
      run_mental_health_pipeline with different n_final_features, n_layers,
      n_epochs, dim_reduction (or embedding_type, reupload, n_repeats)
      occupies its OWN slot in the same dictionary/pickle rather than
      overwriting previously tested configurations (merge_with_existing=True,
      the default for both pipelines).
  12. Before training, both pipelines check whether that portion of the
      experiment has already been executed and saved in the pickle:
      run_pipeline checks the entire configuration, while
      run_mental_health_pipeline checks EACH main disorder individually
      within the same configuration. Anything already trained is
      automatically SKIPPED (no simulation or GridSearchCV time is spent
      again), and only missing work is executed. Use force_rerun=True to
      force retraining even when results already exist.
  13. Includes generate_statistical_tests_table, which compares ALL models
      pairwise using a PAIRED statistical test (Wilcoxon signed-rank or
      paired t-test). Pairing is appropriate because, with n_repeats > 1,
      all models are evaluated on the SAME train/test splits in each
      repetition. The function produces a table with the p-value of each
      pair and whether the difference is statistically significant (that is,
      whether one can state that one model is better than the other or
      whether the result is a tie).

Dependencies:
    pip install pennylane jax jaxlib optax scikit-learn scipy pandas numpy matplotlib
    pip install umap-learn   # optional, only if dim_reduction="umap" is used


Note on backward compatibility:
    Some serialized dictionary keys and legacy output path fragments (for example,
    "repeticoes" and "resumo") are intentionally preserved because existing pickle
    files produced by the original pipeline use those exact keys. Renaming them
    would break the ability to load and resume previous experiments. All Python
    identifiers, documentation, comments, and user-facing messages are translated.

================================================================================
"""

import os

# ==============================================================================
# DETERMINISM / REPRODUCIBILITY — environment variables
# ==============================================================================
# They must be defined BEFORE importing numpy/scipy/sklearn/jax, because
# during import these libraries configure the BLAS backend
# (OpenBLAS/MKL) and the number of threads used in linear-algebra operations.
#
# Why: reduction operations (sums/inner products) executed in
# parallel by multiple threads may be accumulated in DIFFERENT ORDERS
# depending on how many CPU cores are available — and because of
# floating-point rounding, different summation orders may produce
# numerically different results in the last decimal place. This does not change
# the LOGIC of the experiment, but two machines with different numbers of
# cores may obtain slightly different metrics even with
# all seeds fixed. By forcing the linear-algebra libraries to use
# one thread, the operation order no longer depends on the hardware.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
# PYTHONHASHSEED has no effect on THIS process (Python's hash seed is
# fixed when the interpreter starts, before any code runs) —
# but GridSearchCV(n_jobs=-1) and RandomForestClassifier create CHILD PROCESSES
# (through joblib), and each child process IS a new Python interpreter that READS
# this variable during its own initialization. Setting it here ensures that
# child processes also use deterministic hashing.
os.environ.setdefault("PYTHONHASHSEED", "0")

import pickle
import itertools
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
from scipy import stats

import jax
import jax.numpy as jnp
import optax

# double precision: important for numerical stability in quantum simulation
jax.config.update("jax_enable_x64", True)
# Explicitly force the CPU backend. Without this, JAX uses a GPU/TPU
# automatically IF AVAILABLE — and GPU simulation may produce
# numerically different results from CPU simulation (different
# operation orders and different kernels), so two people running the
# same code, one with a GPU and one without, could obtain slightly
# different tables even with identical seeds. Setting "cpu" here ensures that
# ALL environments use exactly the same execution path.
jax.config.update("jax_platform_name", "cpu")

from sklearn.dataset import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_wine
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


# ==============================================================================
# 1. EMBEDDING FUNCTIONS (CLASSICAL -> QUANTUM ENCODING)
# ==============================================================================

def get_n_qubits(n_features, embedding_type):
    """Calculates how many qubits are required for each embedding type."""
    if embedding_type == "amplitude":
        # amplitude embedding requires 2^n_qubits >= n_features
        return max(1, int(np.ceil(np.log2(max(n_features, 2)))))
    elif embedding_type == "angle":
        # angle/phase embedding uses one qubit per feature
        return n_features
    else:
        raise ValueError("embedding_type must be 'amplitude' or 'angle'")


def data_embedding(x, n_qubits, embedding_type="angle"):
    """
    Applies the selected embedding at the beginning of the circuit.

    - amplitude: encodes the feature vector directly into the amplitudes of the
      quantum state (qml.AmplitudeEmbedding). It requires L2 normalization,
      which is performed automatically with normalize=True.
    - angle: also called "phase embedding" in the literature, encodes each
      feature as the angle of an RY rotation on a distinct qubit
      (qml.AngleEmbedding).
    """
    if embedding_type == "amplitude":
        qml.AmplitudeEmbedding(features=x, wires=range(n_qubits),
                                normalize=True, pad_with=0.0)
    elif embedding_type == "angle":
        qml.AngleEmbedding(features=x, wires=range(n_qubits), rotation="Y")
    else:
        raise ValueError("embedding_type must be 'amplitude' or 'angle'")


# ==============================================================================
# 2. ANSÄTZE (VARIATIONAL CIRCUITS) FROM THE LITERATURE
#
# IMPORTANT: each function below applies the variational block of ONE SINGLE
# layer (not the stack of all n_layers). This is what allows the
# main circuit (see VQCBinaryClassifier) to interleave a "reupload" of the
# input data before each layer — the "data re-uploading" technique
# (Pérez-Salinas et al., "Data re-uploading for a universal quantum
# classifier", Quantum 2020), which increases circuit expressivity.
#
# Each function receives:
#   weights_layer : weights for this layer only (a slice of the complete tensor)
#   n_qubits      : number of qubits in the circuit
#   state         : auxiliary state passed between layers (used only by
#                   tree_tensor, which needs to know which qubits are still
#                   "ativos"; os demais ansätze o ignoram e o repassam
#                   inalterado)
# and return the updated `state` (to maintain a uniform interface).
# ==============================================================================

def hardware_efficient_layer(weights_layer, n_qubits, state):
    """
    One layer of the Hardware-Efficient Ansatz (Kandala et al., Nature 2017):
    RY+RZ rotations per qubit followed by a chain of CNOTs.
    weights_layer shape: (n_qubits, 2)
    """
    for q in range(n_qubits):
        qml.RY(weights_layer[q, 0], wires=q)
        qml.RZ(weights_layer[q, 1], wires=q)
    for q in range(n_qubits - 1):
        qml.CNOT(wires=[q, q + 1])
    return state


def strongly_entangling_layer(weights_layer, n_qubits, state):
    """
    One layer of Strongly Entangling Layers (Schuld et al. / PennyLane).
    weights_layer shape: (n_qubits, 3)
    """
    # the template expects a tensor with a layer dimension; we use one layer
    qml.StronglyEntanglingLayers(weights_layer[jnp.newaxis, ...], wires=range(n_qubits))
    return state


def basic_entangler_layer(weights_layer, n_qubits, state):
    """
    One layer of Basic Entangler Layers.
    weights_layer shape: (n_qubits,)
    """
    qml.BasicEntanglerLayers(weights_layer[jnp.newaxis, ...], wires=range(n_qubits))
    return state


def qaoa_inspired_layer(weights_layer, n_qubits, state):
    """
    One layer of the QAOA-inspired ansatz (Farhi & Neven, 2018):
    ZZ cost entanglement through CNOT-RZ-CNOT followed by global RX rotations (mixer).
    weights_layer shape: (2,) -> [gamma, beta]
    """
    gamma, beta = weights_layer[0], weights_layer[1]
    for q in range(n_qubits - 1):
        qml.CNOT(wires=[q, q + 1])
        qml.RZ(gamma, wires=q + 1)
        qml.CNOT(wires=[q, q + 1])
    for q in range(n_qubits):
        qml.RX(beta, wires=q)
    return state


def tree_tensor_layer(weights_layer, n_qubits, state):
    """
    One level of the Tree Tensor Network Ansatz (Grant et al., 2018): blocks of
    2 qubits (RY-RY-CNOT) combine neighboring qubit pairs, reducing the
    set of "active qubits" by half at each tree level.
    weights_layer shape: (ceil(n_qubits/2), 2)
    state: list of active qubits (initialized as range(n_qubits) and
           reduced at each call); passed between layers by the
           main circuit loop.
    """
    active_qubits = state
    new_active = []
    pair_idx = 0
    i = 0
    while i < len(active_qubits):
        if i + 1 < len(active_qubits):
            q0, q1 = active_qubits[i], active_qubits[i + 1]
            qml.RY(weights_layer[pair_idx, 0], wires=q0)
            qml.RY(weights_layer[pair_idx, 1], wires=q1)
            qml.CNOT(wires=[q0, q1])
            new_active.append(q0)
            pair_idx += 1
        else:
            new_active.append(active_qubits[i])
        i += 2
    return new_active


# Central registry: name -> (ONE-layer function, weight-shape function)
# weight_shape always returns the shape of the complete tensor (n_layers, ...); the
# main circuit indexes weights[l] to obtain the weights for each layer.
ANSATZ_REGISTRY = {
    "hardware_efficient": {
        "layer_circuit": hardware_efficient_layer,
        "weight_shape": lambda n_qubits, n_layers: (n_layers, n_qubits, 2),
    },
    "strongly_entangling": {
        "layer_circuit": strongly_entangling_layer,
        "weight_shape": lambda n_qubits, n_layers: qml.StronglyEntanglingLayers.shape(
            n_layers=n_layers, n_wires=n_qubits),
    },
    "basic_entangler": {
        "layer_circuit": basic_entangler_layer,
        "weight_shape": lambda n_qubits, n_layers: qml.BasicEntanglerLayers.shape(
            n_layers=n_layers, n_wires=n_qubits),
    },
    "qaoa_inspired": {
        "layer_circuit": qaoa_inspired_layer,
        "weight_shape": lambda n_qubits, n_layers: (n_layers, 2),
    },
    "tree_tensor": {
        "layer_circuit": tree_tensor_layer,
        "weight_shape": lambda n_qubits, n_layers: (n_layers, max(n_qubits // 2, 1), 2),
    },
}


# ==============================================================================
# 3. SKLEARN-COMPATIBLE BINARY VQC CLASSIFIER
#    (used inside OneVsRestClassifier to create a multiclass classifier)
# ==============================================================================

class VQCBinaryClassifier(BaseEstimator, ClassifierMixin):
    """
    Binary classifier based on a variational quantum circuit.
    It follows the sklearn API (fit/predict/predict_proba/decision_function),
    allowing it to be used directly inside OneVsRestClassifier.

    Circuit structure (data re-uploading):
      Each of the n_layers layers is composed of [data loading + ansatz block].
      In other words, the input data x are re-uploaded at each layer, rather
      than only once at the beginning of the circuit. This increases model
      expressivity, allowing it to approximate more complex functions with
      few qubits (Pérez-Salinas et al., "Data re-uploading for a universal
      quantum classifier", Quantum 2020). Set reupload=False to return to the
      conventional scheme with a single initial data loading step.

    Circuit readout:
      By default (measure_all_qubits=False), the prediction is obtained from
      the PauliZ expectation value on qubit 0 only. With
      measure_all_qubits=True, the PauliZ expectation value is measured on
      ALL n_qubits of the ansatz, and the final prediction is the mean of
      these n_qubits measurements (mean magnetization), keeping the result
      within [-1, 1].

    Efficient training with JAX + Optax:
      - QNode with interface="jax" and diff_method="backprop" -> exact gradient
        through automatic differentiation (much faster than parameter-shift
        in classical simulation).
      - jax.vmap vectorizes the circuit over the entire sample batch (same
        weights, multiple inputs), avoiding a Python sample-by-sample loop.
      - jax.jit compiles the training step (forward + backward + update).
      - optax.adam replaces PennyLane's native optimizer.
    """

    def __init__(self, n_qubits=4, n_layers=2, ansatz="hardware_efficient",
                 embedding_type="angle", reupload=True, n_epochs=60, lr=0.1,
                 batch_size=None, device_name="default.qubit", random_state=42,
                 measure_all_qubits=False):
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.ansatz = ansatz
        self.embedding_type = embedding_type
        self.reupload = reupload  # True = "data re-uploading" in all layers
        self.n_epochs = n_epochs
        self.lr = lr
        self.batch_size = batch_size  # None = batch completo (full-batch GD)
        self.device_name = device_name
        self.random_state = random_state
        # False (default) = measure only qml.PauliZ(0), as before.
        # True = measure qml.PauliZ on ALL n_qubits of the ansatz and aggregate the
        # result using the mean of <Z_i> (it remains within [-1, 1], compatible
        # with _output_to_prob / cost_fn / decision_function unchanged).
        self.measure_all_qubits = measure_all_qubits

    # --------------------------------------------------------------------
    def _build_batched_circuit(self):
        dev = qml.device(self.device_name, wires=self.n_qubits)
        layer_fn = ANSATZ_REGISTRY[self.ansatz]["layer_circuit"]
        embedding_type = self.embedding_type
        n_qubits = self.n_qubits
        n_layers = self.n_layers
        reupload = self.reupload
        measure_all_qubits = self.measure_all_qubits

        @qml.qnode(dev, interface="jax", diff_method="backprop")
        def circuit(x, weights):
            # auxiliary state passed between layers (used only by tree_tensor,
            # which needs to know which qubits remain "active"; the others
            # ansätze o ignoram)
            state = list(range(n_qubits))
            for l in range(n_layers):
                # DATA RE-UPLOADING: one layer = one data-loading operation +
                # one ansatz block. The input data x are re-uploaded
                # before each of the n_layers layers (instead of a single
                # loading at the beginning), which increases circuit expressivity
                # (Pérez-Salinas et al., Quantum 2020). If reupload=False,
                # the embedding is applied only in the first layer (the
                # conventional behavior of a single initial loading step).
                if reupload or l == 0:
                    data_embedding(x, n_qubits, embedding_type)
                state = layer_fn(weights[l], n_qubits, state)
            # READOUT: by default (measure_all_qubits=False), measure
            # only qml.PauliZ(0), as in the original behavior. When
            # measure_all_qubits=True, measure qml.PauliZ on ALL n_qubits
            # of the ansatz (not only the first one), returning a tuple with one
            # measurement per qubit -- aggregation of these measurements into a single
            # scalar per sample is performed immediately below, outside the QNode.
            if measure_all_qubits:
                return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]
            return qml.expval(qml.PauliZ(0))

        # vectorize the circuit over axis 0 of x (batch); the weights (axis 1
        # of the arguments) are shared by all samples (in_axes=None)
        vmapped_circuit = jax.vmap(circuit, in_axes=(0, None))

        if not measure_all_qubits:
            return vmapped_circuit

        # When measure_all_qubits=True, the QNode returns a list of
        # n_qubits measurements; after vmap, this becomes a tuple of n_qubits
        # arrays, each with shape (batch,). We stack them into (batch, n_qubits)
        # and aggregate them using the MEAN of <Z_i> -- the result remains a
        # scalar per sample in [-1, 1] (mean magnetization), preserving
        # the interface expected by _output_to_prob / cost_fn /
        # decision_function without requiring any other pipeline changes.
        def batched_circuit_all_qubits(x, weights):
            expvals_per_qubit = jnp.stack(vmapped_circuit(x, weights), axis=-1)
            return jnp.mean(expvals_per_qubit, axis=-1)

        return batched_circuit_all_qubits

    @staticmethod
    def _output_to_prob(expval):
        """Maps <Z> from [-1, 1] to a probability in [0, 1]."""
        return (expval + 1.0) / 2.0

    # --------------------------------------------------------------------
    def fit(self, X, y):
        X = jnp.asarray(np.asarray(X, dtype=np.float64))
        y = jnp.asarray(np.asarray(y, dtype=np.float64))  # 0/1 (provided by OvR)

        batched_circuit = self._build_batched_circuit()
        self.batched_circuit_ = batched_circuit
        shape = ANSATZ_REGISTRY[self.ansatz]["weight_shape"](self.n_qubits, self.n_layers)

        key = jax.random.PRNGKey(self.random_state)
        weights = jax.random.uniform(key, shape=shape, minval=0.0, maxval=2 * jnp.pi)

        optimizer = optax.adam(learning_rate=self.lr)
        opt_state = optimizer.init(weights)

        def cost_fn(w, X_batch, y_batch):
            expvals = batched_circuit(X_batch, w)
            preds = jnp.clip(self._output_to_prob(expvals), 1e-7, 1 - 1e-7)
            return -jnp.mean(y_batch * jnp.log(preds) + (1 - y_batch) * jnp.log(1 - preds))

        @jax.jit
        def update_step(w, opt_state, X_batch, y_batch):
            loss, grads = jax.value_and_grad(cost_fn)(w, X_batch, y_batch)
            updates, opt_state = optimizer.update(grads, opt_state, w)
            w = optax.apply_updates(w, updates)
            return w, opt_state, loss

        n_samples = X.shape[0]
        rng = np.random.default_rng(self.random_state)

        # fixed batch size (drop_last) to keep static shapes and
        # aproveitar o jit; batch_size=None -> gradiente descendente full-batch,
        # is commonly used in VQCs because of simulation cost.
        bs = n_samples if self.batch_size is None else min(self.batch_size, n_samples)
        n_batches = max(n_samples // bs, 1)

        for epoch in range(self.n_epochs):
            perm = rng.permutation(n_samples)
            for b in range(n_batches):
                batch_idx = perm[b * bs:(b + 1) * bs]
                X_batch, y_batch = X[batch_idx], y[batch_idx]
                weights, opt_state, loss = update_step(weights, opt_state, X_batch, y_batch)

        self.weights_ = weights
        self.classes_ = np.array([0, 1])
        return self

    # --------------------------------------------------------------------
    def decision_function(self, X):
        """Returns the raw <Z> expectation value (range [-1, 1])."""
        X = jnp.asarray(np.asarray(X, dtype=np.float64))
        expvals = self.batched_circuit_(X, self.weights_)
        return np.asarray(expvals)

    def predict_proba(self, X):
        p1 = self._output_to_prob(self.decision_function(X))
        return np.column_stack([1 - p1, p1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ==============================================================================
# 4. DATA LOADING, DIMENSIONALITY REDUCTION, AND PREPROCESSING
# ==============================================================================

def create_reducer(method="pca", n_components=4, random_state=42):
    """Creates, but does NOT fit, the dimensionality reducer."""
    if method == "pca":
        return PCA(
            n_components=n_components,
            random_state=random_state,
            svd_solver="full",
        )
    elif method == "umap":
        try:
            import umap
        except ImportError as e:
            raise ImportError(
                "The 'umap' method requires the 'umap-learn' package "
                "(pip install umap-learn)."
            ) from e
        return umap.UMAP(
            n_components=n_components,
            random_state=random_state,
            n_jobs=1,
        )
    else:
        raise ValueError("method must be 'pca' or 'umap'")


def reduce_dimensionality(X, method="pca", n_components=4, random_state=42):
    """Compatibility helper: fits the reducer on X itself and transforms it.

    IMPORTANT: this function must NOT be used to prepare data before a
    train/test split. In the experimental pipelines, use
    ``preprocess_quantum_split`` or the classical ``Pipeline`` below, which
    fit the reducer only on the appropriate training data.
    """
    reducer = create_reducer(method, n_components, random_state)
    X_reduzido = reducer.fit_transform(X)
    return X_reduzido, reducer


def preprocess_quantum_split(
    X_train,
    X_test,
    embedding_type="angle",
    dim_reduction=None,
    n_final_features=None,
    max_qubits=6,
    random_state=42,
):
    """Fits dimensionality reduction + scaling ONLY on the training data and
    transforms the test data.

    For ``angle``, the scaler is MinMax over [0, pi]. For ``amplitude``,
    StandardScaler is used (AmplitudeEmbedding performs L2 normalization internally).
    """
    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)

    reducer = None
    if dim_reduction is not None:
        if n_final_features is None:
            raise ValueError(
                "Defina n_features_final ao usar reducao_dim='pca' ou 'umap'."
            )
        reducer = create_reducer(
            dim_reduction,
            n_components=n_final_features,
            random_state=random_state,
        )
        X_train_proc = reducer.fit_transform(X_train)
        X_test_proc = reducer.transform(X_test)
    elif embedding_type == "angle" and X_train.shape[1] > max_qubits:
        reducer = create_reducer(
            "pca",
            n_components=max_qubits,
            random_state=random_state,
        )
        X_train_proc = reducer.fit_transform(X_train)
        X_test_proc = reducer.transform(X_test)
    else:
        X_train_proc = X_train.copy()
        X_test_proc = X_test.copy()

    if embedding_type == "angle":
        scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)
    elif embedding_type == "amplitude":
        scaler = StandardScaler()
    else:
        raise ValueError("embedding_type must be 'amplitude' or 'angle'")

    X_train_proc = scaler.fit_transform(X_train_proc)
    X_test_proc = scaler.transform(X_test_proc)
    return X_train_proc, X_test_proc, reducer, scaler


def create_classical_pipeline(
    estimator,
    param_grid,
    dim_reduction=None,
    n_final_features=None,
    random_state=42,
):
    """Creates a classical leakage-free Pipeline for use inside GridSearchCV.

    In each fold, GridSearchCV itself fits ``reducer`` and
    ``MinMaxScaler([0,1])`` only on the training sub-fold and applies the
    transformations to the validation fold. During the final refit, these
    steps are fitted on the entire outer-training set and only then applied
    to the outer-test set.
    """
    steps = []
    if dim_reduction is not None:
        if n_final_features is None:
            raise ValueError(
                "Defina n_features_final ao usar reducao_dim='pca' ou 'umap'."
            )
        steps.append((
            "reducer",
            create_reducer(
                dim_reduction,
                n_components=n_final_features,
                random_state=random_state,
            ),
        ))

    steps.append(("scaler", MinMaxScaler(feature_range=(0, 1), clip=True)))
    steps.append(("model", estimator))
    pipeline = Pipeline(steps)

    param_grid_pipeline = {
        f"model__{key}": values for key, values in param_grid.items()
    }
    return pipeline, param_grid_pipeline


def preprocess_features(X, embedding_type="angle", dim_reduction=None,
                           n_final_features=None, max_qubits=6,
                           random_state=42):
    """DEPRECATED for experimental evaluation.

    Kept only for compatibility with external uses. It fits the transformations
    on X itself and therefore must NOT be used before a train/test split in
    evaluation experiments. The main pipelines now use split-aware,
    leakage-free preprocessing.
    """
    X = np.asarray(X, dtype=float)
    reducer = None
    if dim_reduction is not None:
        if n_final_features is None:
            raise ValueError(
                "Defina n_features_final ao usar reducao_dim='pca' ou 'umap'."
            )
        X, reducer = reduce_dimensionality(
            X, method=dim_reduction, n_components=n_final_features,
            random_state=random_state,
        )
    elif embedding_type == "angle" and X.shape[1] > max_qubits:
        X, reducer = reduce_dimensionality(
            X, method="pca", n_components=max_qubits,
            random_state=random_state,
        )

    if embedding_type == "angle":
        X = MinMaxScaler(feature_range=(0, np.pi), clip=True).fit_transform(X)
    elif embedding_type == "amplitude":
        X = StandardScaler().fit_transform(X)
    else:
        raise ValueError("embedding_type must be 'amplitude' or 'angle'")
    return X, reducer

def load_data(dataset_name="iris", embedding_type="angle",
                    dim_reduction=None, n_final_features=None,
                    max_qubits=6, random_state=42):
    """Loads Iris/Wine WITHOUT fitting preprocessing on the complete dataset.

    Preprocessing arguments are retained for compatibility, but dimensionality
    reduction and scaling are performed only after the train/test split.
    """
    if dataset_name == "iris":
        data = load_iris()
    elif dataset_name == "wine":
        data = load_wine()
    else:
        raise ValueError("dataset_name must be 'iris' or 'wine'")

    X = np.asarray(data.data, dtype=float)
    y = np.asarray(data.target)
    return X, y, data.target_names, None


# ==============================================================================
# 4B. LOADING: MENTAL-HEALTH DATASET (multi-disorder)
#
# This dataset has a "main disorder" column (main.disorder) that
# is subdivided into "specific disorders" (specific.disorder), in addition to a
# healthy control group ("Healthy control"). For EACH main disorder,
# we train a MULTICLASS classifier whose classes are the
# specific disorders associated with that main disorder + the healthy control group
# (e.g., for "Mood disorder", the classes could be "Bipolar disorder",
# "Depressive disorder", and "Healthy control").
#
# Note: unlike the reference code (which used main.disorder as a binary
# target), here the target y is always specific.disorder — this produces the
# requested multiclass problem "specific disorders + control".
# ==============================================================================

def load_mental_health_csv(csv_path):
    """Loads the mental-health dataset CSV and prints an initial summary."""
    df = pd.read_csv(csv_path)
    print(f"Base carregada: {csv_path}")
    print(f"Shape: {df.shape}")
    print(df.head())
    return df


def list_main_disorders(df, column_main="main.disorder",
                                 group_control="Healthy control"):
    """
    Returns the list of main disorders present in the dataset, EXCLUDING the
    healthy control group (which is not a disorder to be predicted, but rather
    the reference group included in each training subset).
    """
    return [d for d in df[column_main].unique() if d != group_control]


def disorder_summary(df, column_main="main.disorder",
                      column_specific="specific.disorder",
                      group_control="Healthy control"):
    """
    For each main disorder, prints the specific disorders that compose it and
    the sample count for each one — useful for inspecting the dataset before
    starting training.
    """
    for disorder in list_main_disorders(df, column_main, group_control):
        subset = df[df[column_main] == disorder]
        counts = subset[column_specific].value_counts()
        print(f"\n{disorder} (n={len(subset)})")
        for specific, count in counts.items():
            print(f"   - {specific}: {count}")

    n_control = int((df[column_main] == group_control).sum())
    print(f"\n{group_control} (n={n_control})")


def prepare_disorder_data(df, main_disorder,
                             column_main="main.disorder",
                             column_specific="specific.disorder",
                             group_control="Healthy control",
                             columns_exclude=("sex_M",),
                             remove_rows_with_nulls=True):
    """
    Builds X and y for the multiclass problem associated with ONE main disorder:
    the classes are the specific disorders of that main disorder PLUS the
    healthy control group.

    Parameters
    ----------
    df : complete dataset DataFrame
    main_disorder : value of `main_column` to isolate (e.g., "Mood disorder")
    main_column, specific_column : names of the label columns in the dataset
    control_group : value of `main_column` identifying the healthy group
    columns_to_exclude : other columns to remove from X (non-features), in
                         addition to main_column and specific_column; by default,
                         reproduces the reference code by removing "sex_M"
    remove_rows_with_nulls : if True (default), remove rows containing any
        null value in X (VQC and SVM do not handle NaN); if False, only warn

    Returns
    -------
    X : np.ndarray (float), shape (n_samples, n_features)
    y_encoded : np.ndarray (int), encoded labels
    class_names : list of class names, in the order used by
                  LabelEncoder (i.e., class_names[i] == the original label
                  for the class encoded as i)
    """
    mask = (df[column_main] == main_disorder) | (df[column_main] == group_control)
    df_subset = df[mask].copy()

    specific_disorders = list(df_subset[column_specific].unique())
    print(f"\nMain disorder: {main_disorder}")
    print(f"Classes (specific disorders + control): {specific_disorders}")
    print(f"Total samples: {len(df_subset)}")

    columns_to_remove = [column_main, column_specific, *columns_exclude]
    columns_to_remove = [c for c in columns_to_remove if c in df_subset.columns]

    X_df = df_subset.drop(columns=columns_to_remove)
    y_raw = df_subset[column_specific]

    # additional safeguard: any remaining non-numeric column (e.g., an ID,
    # a date) cannot become a quantum feature — it is removed with a warning,
    # instead of breaking .to_numpy(dtype=float) below. If any of these
    # columns is expected, include it explicitly in columns_to_exclude.
    non_numeric_columns = X_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_columns:
        print(f"Warning: removing non-numeric columns from X: {non_numeric_columns}")
        X_df = X_df.drop(columns=non_numeric_columns)

    n_nulls = int(X_df.isnull().values.sum())
    if n_nulls > 0:
        print(f"Warning: {n_nulls} null values found in X.")
        if remove_rows_with_nulls:
            valid_rows = ~X_df.isnull().any(axis=1)
            print(f"Removing {int((~valid_rows).sum())} rows with null values.")
            X_df = X_df[valid_rows]
            y_raw = y_raw[valid_rows]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_raw)
    class_names = list(label_encoder.classes_)

    X = X_df.to_numpy(dtype=float)
    y_encoded = np.asarray(y_encoded)

    return X, y_encoded, class_names


# ==============================================================================
# 5. TRAINING + EVALUATION BY ANSATZ
# ==============================================================================

def train_and_evaluate(ansatz_name, X_train, X_test, y_train, y_test,
                       n_qubits, n_layers=2, embedding_type="angle",
                       reupload=True, n_epochs=60, lr=0.1, batch_size=None,
                       random_state=42, measure_all_qubits=False):
    """
    Trains the OneVsRestClassifier (one binary VQC per class) for a given
    ansatz and computes a complete set of evaluation metrics, in addition to
    extracting the trained parameters (weights) of each circuit.

    measure_all_qubits : bool, default False
        If True, the circuit readout considers ALL n_qubits of the ansatz
        (mean of <Z_i>) instead of only qubit 0 — see VQCBinaryClassifier.
    """
    dataset_clf = VQCBinaryClassifier(
        n_qubits=n_qubits,
        n_layers=n_layers,
        ansatz=ansatz_name,
        embedding_type=embedding_type,
        reupload=reupload,
        n_epochs=n_epochs,
        lr=lr,
        batch_size=batch_size,
        random_state=random_state,
        measure_all_qubits=measure_all_qubits,
    )

    ovr_clf = OneVsRestClassifier(dataset_clf)
    ovr_clf.fit(X_train, y_train)

    y_pred = ovr_clf.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        # per-class metrics (individual precision/recall/f1/support)
        "classification_report": classification_report(
            y_test, y_pred, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "y_true": np.asarray(y_test),
        "y_pred": np.asarray(y_pred),
    }

    # -------------------------------------------------------------------
    # Trained parameters: OneVsRestClassifier keeps one VQCBinaryClassifier
    # per class in ovr_clf.estimators_ (in the same order as ovr_clf.classes_).
    # Convert the weights (JAX arrays) to plain NumPy arrays, ensuring that the
    # final dictionary can be serialized as a pickle without depending on JAX.
    # -------------------------------------------------------------------
    params_per_class = {}
    for class_idx, estimator in zip(ovr_clf.classes_, ovr_clf.estimators_):
        params_per_class[int(class_idx)] = np.asarray(estimator.weights_)

    metrics["params"] = params_per_class
    metrics["config"] = {
        "ansatz": ansatz_name,
        "n_qubits": n_qubits,
        "n_layers": n_layers,
        "embedding_type": embedding_type,
        "reupload": reupload,
        "n_epochs": n_epochs,
        "lr": lr,
        "batch_size": batch_size,
        "random_state": random_state,
        "measure_all_qubits": measure_all_qubits,
        "weight_shape_por_classe": {
            class_idx: params_per_class[class_idx].shape for class_idx in params_per_class
        },
    }

    return ovr_clf, metrics


# ==============================================================================
# 6. CLASSICAL BASELINES WITH GRID SEARCH (sklearn)
#
# Train/evaluate classical models on the SAME training and test sets
# used by the VQCs, allowing direct comparison of quantum versus
# classical performance under the same conditions. GridSearchCV with
# stratified cross-validation is used to optimize each model's hyperparameters.
#
# Included models:
#   - SVM (sklearn.svm.SVC), one grid per kernel (linear, rbf, sigmoid, poly)
#   - Decision Tree (sklearn.tree.DecisionTreeClassifier)
#   - MLP / neural network (sklearn.neural_network.MLPClassifier), with a grid
#     of architectures/hyperparameters designed to provide a
#     well-tuned configuration (early stopping, multiple layer depths/widths)
#   - Random Forest (sklearn.ensemble.RandomForestClassifier) — an additional
#     additional baseline: it is often one of the strongest classifiers and
#     robust for tabular data (such as the data in this pipeline), with little
#     tuning required and good resistance to overfitting through bagging
# ==============================================================================

SVM_PARAM_GRIDS = {
    "linear": {"kernel": ["linear"], "C": [0.1, 1, 10, 100]},
    "rbf": {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "gamma": [0.001, 0.01, 0.1, 1],
    },
    "sigmoid": {
        "kernel": ["sigmoid"],
        "C": [0.1, 1, 10, 100],
        "gamma": [0.001, 0.01, 0.1, 1],
    },
    "poly": {
        "kernel": ["poly"],
        "C": [0.1, 1, 10, 100],
        "degree": [2, 3, 4],
        "gamma": [0.001, 0.01, 0.1, 1],
    },
}

# Decision Tree hyperparameter grid
DECISION_TREE_PARAM_GRID = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

# MLP (neural-network) hyperparameter grid. A high max_iter + early_stopping
# ensure that each configuration trains until it actually converges (or stops early
# because validation stabilizes), rather than stopping because it runs out of iterations —
# this is what gives the MLP selected by
# GridSearchCV a well-tuned final configuration rather than an arbitrary fixed architecture.
MLP_PARAM_GRID = {
    "hidden_layer_sizes": [(64,), (128, 64), (100, 50, 25), (64, 64, 64)],
    "activation": ["relu", "tanh"],
    "alpha": [1e-4, 1e-3, 1e-2],
    "learning_rate_init": [1e-3, 1e-2],
}

# Random Forest hyperparameter grid
RANDOM_FOREST_PARAM_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2"],
}

# Registry of single-model classical baselines (i.e., excluding SVM,
# which has one grid per kernel and is handled separately through SVM_PARAM_GRIDS). Each
# entry specifies how to build the estimator (random_state function for
# reproducibility) and the hyperparameter grid to optimize through GridSearchCV.
CLASSICAL_MODELS_REGISTRY = {
    "decision_tree": {
        "estimator_fn": lambda random_state: DecisionTreeClassifier(random_state=random_state),
        "param_grid": DECISION_TREE_PARAM_GRID,
    },
    "mlp": {
        "estimator_fn": lambda random_state: MLPClassifier(
            solver="adam", max_iter=3000, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=15,
            random_state=random_state,
        ),
        "param_grid": MLP_PARAM_GRID,
    },
    "random_forest": {
        # n_jobs=1 (NOT -1) is intentional and critical for reproducibility:
        # GridSearchCV already runs with n_jobs=-1 (parallelizing grid
        # candidates). If RandomForestClassifier ALSO used n_jobs=-1, this would create
        # NESTED parallelism (GridSearchCV child processes each
        # attempting to create their own child processes for the trees) —
        # this is a documented scikit-learn/joblib issue (see
        # scikit-learn/scikit-learn#22230): even with a fixed random_state, the
        # way each tree's seeds are distributed among
        # processes becomes non-deterministic in this nested scenario,
        # causing Random Forest to produce DIFFERENT results on each run
        # despite a fixed random_state. With n_jobs=1 here (only GridSearchCV
        # is parallelized), the result is always the same.
        "estimator_fn": lambda random_state: RandomForestClassifier(
            random_state=random_state, n_jobs=1,
        ),
        "param_grid": RANDOM_FOREST_PARAM_GRID,
    },
}


def train_evaluate_svm(kernel, X_train, X_test, y_train, y_test,
                         cv_splits=5, random_state=42,
                         dim_reduction=None, n_final_features=None):
    """
    Trains an SVM (sklearn.svm.SVC) with GridSearchCV for the specified kernel,
    using StratifiedKFold, and evaluates it on the test set. Returns a
    metrics dictionary in the SAME format used for the VQCs
    (see train_and_evaluate), enabling direct comparison between them and
    joint storage in the same results pickle.
    """
    if kernel not in SVM_PARAM_GRIDS:
        raise ValueError(f"kernel must be one of {list(SVM_PARAM_GRIDS.keys())}")

    return _train_evaluate_generic_classical(
        # random_state is passed explicitly for clarity/robustness: because
        # With probability=False (the default, never changed in this pipeline), SVC
        # is already deterministic without random_state — but setting it here documents
        # the intention and protects against regression if probability=True is
        # enabled in the future (in that case SVC uses RNG internally for
        # probability estimates through Platt cross-validation).
        SVC(random_state=random_state), SVM_PARAM_GRIDS[kernel], X_train, X_test, y_train, y_test,
        cv_splits=cv_splits, random_state=random_state,
        config_extra={"kernel": kernel},
        dim_reduction=dim_reduction, n_final_features=n_final_features,
    )


def train_evaluate_classical(model_name, X_train, X_test, y_train, y_test,
                              cv_splits=5, random_state=42,
                              dim_reduction=None, n_final_features=None):
    """
    Trains a classical baseline from CLASSICAL_MODELS_REGISTRY (currently:
    "decision_tree", "mlp", or "random_forest") with GridSearchCV using
    StratifiedKFold, and evaluates it on the test set. Returns a metrics
    dictionary in the SAME format used for the VQCs and SVM (see
    train_and_evaluate / train_evaluate_svm).
    """
    if model_name not in CLASSICAL_MODELS_REGISTRY:
        raise ValueError(f"model_name must be one of {list(CLASSICAL_MODELS_REGISTRY.keys())}")

    entry = CLASSICAL_MODELS_REGISTRY[model_name]
    estimator = entry["estimator_fn"](random_state)
    return _train_evaluate_generic_classical(
        estimator, entry["param_grid"], X_train, X_test, y_train, y_test,
        cv_splits=cv_splits, random_state=random_state,
        config_extra={"modelo": model_name},
        dim_reduction=dim_reduction, n_final_features=n_final_features,
    )


def _train_evaluate_generic_classical(
    estimator,
    param_grid,
    X_train,
    X_test,
    y_train,
    y_test,
    cv_splits=5,
    random_state=42,
    config_extra=None,
    dim_reduction=None,
    n_final_features=None,
):
    """Classical GridSearchCV with preprocessing performed entirely within the folds."""
    cv = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=random_state,
    )

    pipeline, param_grid_pipeline = create_classical_pipeline(
        estimator,
        param_grid,
        dim_reduction=dim_reduction,
        n_final_features=n_final_features,
        random_state=random_state,
    )

    grid = GridSearchCV(
        pipeline,
        param_grid_pipeline,
        scoring="f1_weighted",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    grid.fit(X_train, y_train)

    best_pipeline = grid.best_estimator_
    model = best_pipeline.named_steps["model"]
    y_pred = best_pipeline.predict(X_test)

    best_params_model = {
        key.replace("model__", "", 1): value
        for key, value in grid.best_params_.items()
    }

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "classification_report": classification_report(
            y_test, y_pred, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "y_true": np.asarray(y_test),
        "y_pred": np.asarray(y_pred),
        "best_params": best_params_model,
        "model": model,
        "pipeline": best_pipeline,
    }

    metrics["params"] = _extract_classical_parameters(model)
    metrics["config"] = {
        "cv_splits": cv_splits,
        "random_state": random_state,
        "param_grid": param_grid,
        "best_params": best_params_model,
        "reducao_dim": dim_reduction,
        "n_features_final": n_final_features,
        "scaler": "MinMaxScaler[0,1]",
        **(config_extra or {}),
    }
    return metrics


def _extract_classical_parameters(model):
    """
    Extracts the relevant trained parameters from an already-fitted classical
    sklearn estimator, covering the model types used in this pipeline (SVM,
    Decision Tree, MLP, Random Forest). Always returns plain NumPy arrays
    (never JAX/sklearn objects) to keep the final pickle portable.
    """
    if isinstance(model, SVC):
        return {
            "support_vectors_": np.asarray(model.support_vectors_),
            "n_support_": np.asarray(model.n_support_),
            "dual_coef_": np.asarray(model.dual_coef_),
            "intercept_": np.asarray(model.intercept_),
        }
    if isinstance(model, DecisionTreeClassifier):
        return {
            "feature_importances_": np.asarray(model.feature_importances_),
            "tree_depth": model.get_depth(),
            "n_leaves": model.get_n_leaves(),
        }
    if isinstance(model, MLPClassifier):
        return {
            "coefs_": [np.asarray(c) for c in model.coefs_],
            "intercepts_": [np.asarray(b) for b in model.intercepts_],
            "n_iter_": model.n_iter_,
        }
    if isinstance(model, RandomForestClassifier):
        return {
            "feature_importances_": np.asarray(model.feature_importances_),
            "n_estimators": model.n_estimators,
        }
    return {}


def run_svm_baseline(X_train, X_test, y_train, y_test,
                           kernels=("linear", "rbf", "poly", "sigmoid"),
                           cv_splits=5, random_state=42,
                           dim_reduction=None, n_final_features=None):
    """
    Runs train_evaluate_svm for each kernel in `kernels`, using the SAME
    training/test set as the VQC pipeline.

    Returns a dictionary {"svm_<kernel>": metrics, ...} in the same format
    as the VQC results, ready to be merged with `results` in
    run_pipeline (or used independently).
    """
    results_svm = {}
    for kernel in kernels:
        print(f"\n=== Training SVM (GridSearchCV) with kernel: {kernel} ===")
        metrics = train_evaluate_svm(
            kernel, X_train, X_test, y_train, y_test,
            cv_splits=cv_splits, random_state=random_state,
            dim_reduction=dim_reduction, n_final_features=n_final_features,
        )
        results_svm[f"svm_{kernel}"] = metrics

        print(f"Best hyperparameters: {metrics['best_params']}")
        print(f"Accuracy:             {metrics['accuracy']:.4f}")
        print(f"Precision (weighted): {metrics['precision_weighted']:.4f}")
        print(f"Recall (weighted):    {metrics['recall_weighted']:.4f}")
        print(f"F1-score (weighted):  {metrics['f1_weighted']:.4f}")

    return results_svm


def run_classical_baselines(X_train, X_test, y_train, y_test,
                                  include_svm=True,
                                  svm_kernels=("linear", "rbf", "poly", "sigmoid"),
                                  include_decision_tree=True,
                                  include_mlp=True,
                                  include_random_forest=True,
                                  cv_splits=5, random_state=42,
                                  dim_reduction=None, n_final_features=None):
    """
    Trains/evaluates ALL enabled classical baselines (SVM, Decision Tree,
    MLP, and Random Forest) on the SAME training/test set as the VQC
    pipeline, each with its own GridSearchCV.

    Returns a single dictionary combining all of them:
        {"svm_<kernel>": {...}, "decision_tree": {...}, "mlp": {...},
         "random_forest": {...}}
    (only those enabled through include_* are present), ready to be
    merged with the VQC results.
    """
    results = {}

    if include_svm:
        results.update(run_svm_baseline(
            X_train, X_test, y_train, y_test,
            kernels=svm_kernels, cv_splits=cv_splits, random_state=random_state,
            dim_reduction=dim_reduction, n_final_features=n_final_features,
        ))

    models_to_include = {
        "decision_tree": include_decision_tree,
        "mlp": include_mlp,
        "random_forest": include_random_forest,
    }
    display_names = {
        "decision_tree": "Decision Tree",
        "mlp": "MLP (rede neural)",
        "random_forest": "Random Forest",
    }
    for model_name, include in models_to_include.items():
        if not include:
            continue
        print(f"\n=== Training {display_names[model_name]} (GridSearchCV) ===")
        metrics = train_evaluate_classical(
            model_name, X_train, X_test, y_train, y_test,
            cv_splits=cv_splits, random_state=random_state,
            dim_reduction=dim_reduction, n_final_features=n_final_features,
        )
        results[model_name] = metrics

        print(f"Best hyperparameters: {metrics['best_params']}")
        print(f"Accuracy:             {metrics['accuracy']:.4f}")
        print(f"Precision (weighted): {metrics['precision_weighted']:.4f}")
        print(f"Recall (weighted):    {metrics['recall_weighted']:.4f}")
        print(f"F1-score (weighted):  {metrics['f1_weighted']:.4f}")

    return results


def aggregate_repetitions(metrics_list):
    """
    Aggregates a list of metric dictionaries (one entry per experiment repetition,
    in the format returned by train_and_evaluate / train_evaluate_svm) into a
    summary containing the mean and standard deviation of the main scalar metrics.
    """
    scalar_keys = [
        "accuracy",
        "precision_macro", "precision_weighted",
        "recall_macro", "recall_weighted",
        "f1_macro", "f1_weighted",
    ]
    summary = {"n_repeats": len(metrics_list)}
    for key in scalar_keys:
        values = np.array([m[key] for m in metrics_list], dtype=float)
        summary[f"{key}_mean"] = float(values.mean())
        summary[f"{key}_std"] = float(values.std())
    return summary


def plot_results(results, output_path="/mnt/user-data/outputs/comparacao_ansatze.png",
                     title="Comparison of Ansätze (VQC) vs. Classical Baselines"):
    """
    Plots the accuracy of each model. It works both for a single-run format
    (results[name]["accuracy"]) and for the aggregated multiple-repetition
    format (results[name]["resumo"]["accuracy_mean"/"accuracy_std"], see
    train_evaluate_all_models with n_repeats>1). In the latter case, error
    bars represent the standard deviation across repetitions.
    """
    names = list(results.keys())
    values, errors = [], []
    for name in names:
        entry = results[name]
        if "resumo" in entry:
            values.append(entry["resumo"]["accuracy_mean"])
            errors.append(entry["resumo"]["accuracy_std"])
        else:
            values.append(entry["accuracy"])
            errors.append(0.0)

    # visually distinguish classical baselines (orange) from VQCs (blue),
    # using the same model-type classification used in the comparison table
    colors = ["steelblue" if _model_type(name) == "VQC" else "darkorange" for name in names]
    use_error_bars = any(e > 0 for e in errors)

    plt.figure(figsize=(9, 5))
    plt.bar(names, values, yerr=errors if use_error_bars else None,
            capsize=4, color=colors)
    plt.ylabel("Accuracy" + (" (mean ± standard deviation)" if use_error_bars else ""))
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f"\nComparison plot saved to: {output_path}")


# ------------------------------------------------------------------------
# Helper: reconstruct {model_name: [metrics_rep_0, ...]} from a
# previously saved disorder result, enabling partial resumption.
# ------------------------------------------------------------------------
def _extract_partial_repetitions(disorder_data, model_names):
    """
    Returns {model_name: metrics_list_per_repetition} if `disorder_data` is in
    aggregated format (n_repeats > 1, with "repeticoes" in each model) and
    all models have the SAME number of saved repetitions. Returns None if the
    format is flat (n_repeats == 1 in the pickle), if any model is missing,
    or if the number of repetitions differs across models (which should not
    occur because the checkpoint is saved only after an ENTIRE repetition finishes).
    """
    if not disorder_data:
        return None
    repetitions = {}
    sizes = set()
    for name in model_names:
        entry = disorder_data.get(name)
        if not isinstance(entry, dict) or "repeticoes" not in entry:
            return None
        repetitions[name] = list(entry["repeticoes"])
        sizes.add(len(entry["repeticoes"]))
    if len(sizes) != 1:
        return None
    return repetitions


def _all_model_names(include_svm_baseline, svm_kernels, include_decision_tree,
                          include_mlp, include_random_forest):
    ansatz_names = list(ANSATZ_REGISTRY.keys())
    names_svm = [f"svm_{k}" for k in svm_kernels] if include_svm_baseline else []
    names_classical_extra = [
        name for name, include in (
            ("decision_tree", include_decision_tree),
            ("mlp", include_mlp),
            ("random_forest", include_random_forest),
        ) if include
    ]
    return ansatz_names + names_svm + names_classical_extra


def train_evaluate_all_models(X, y, class_names,
                                   n_layers=2, reupload=True, n_epochs=60,
                                   lr=0.1, batch_size=None,
                                   embedding_type="angle",
                                   dim_reduction=None, n_final_features=None,
                                   max_qubits=6,
                                   test_size=0.3, random_state=42,
                                   n_repeats=1,
                                   include_svm_baseline=True,
                                   svm_kernels=("linear", "rbf", "poly", "sigmoid"),
                                   svm_cv_splits=5,
                                   include_decision_tree=True,
                                   include_mlp=True,
                                   include_random_forest=True,
                                   plot_path=None,
                                   title="Comparison of Ansätze (VQC) vs. Classical Baselines",
                                   measure_all_qubits=False,
                                   existing_repetitions=None,
                                   callback_checkpoint=None):
    """
    Same behavior as the original version (see the previous docstring), with
    two new parameters:

    existing_repetitions : optional dict {model_name: [metrics_rep_0, ...]}
        If provided, training RESUMES from repetition
        `len(existing_repetitions[<any_model>])` instead of 0. Existing
        repetitions are reused exactly as they are, without retraining.
    callback_checkpoint : optional function (repetitions_per_model, completed_index) -> None
        If provided, it is called immediately after EACH `completed_index`
        repetition finishes (all ansätze + all classical baselines for that
        repetition), receiving the CURRENT and COMPLETE state of
        `repetitions_per_model` (including previous repetitions). Use this to
        persist progress incrementally — this is how
        `run_mental_health_pipeline`, below, implements per-repetition checkpointing.
    """
    ansatz_names = list(ANSATZ_REGISTRY.keys())
    names_svm = [f"svm_{k}" for k in svm_kernels] if include_svm_baseline else []
    names_classical_extra = [
        name for name, include in (
            ("decision_tree", include_decision_tree),
            ("mlp", include_mlp),
            ("random_forest", include_random_forest),
        ) if include
    ]
    classical_names = names_svm + names_classical_extra
    all_names = ansatz_names + classical_names

    if existing_repetitions:
        repetitions_per_model = {
            name: list(existing_repetitions.get(name, [])) for name in all_names
        }
        start_index = len(repetitions_per_model[all_names[0]])
        if start_index > 0:
            print(f"[resume] {start_index} repetition(s) already completed "
                  f"previously — resuming from repetition "
                  f"{start_index + 1}/{n_repeats}.")
    else:
        repetitions_per_model = {name: [] for name in all_names}
        start_index = 0

    for i in range(start_index, n_repeats):
        random_state_i = random_state + i
        if n_repeats > 1:
            print(f"\n{'#' * 80}\nREPETITION {i + 1}/{n_repeats} "
                  f"(random_state={random_state_i})\n{'#' * 80}")

        # SAME training/test split for this repetition, used by both the
        # VQCs and ALL classical baselines below.
        X_train_raw, X_test_raw, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state_i, stratify=y
        )

        # QUANTUM BRANCH: fit PCA/UMAP and MinMax[0,pi] only on the
        # outer-training set for this repetition; the outer-test receives only
        # transform(). This eliminates any information leakage.
        X_train_q, X_test_q, reducer_q, scaler_q = preprocess_quantum_split(
            X_train_raw,
            X_test_raw,
            embedding_type=embedding_type,
            dim_reduction=dim_reduction,
            n_final_features=n_final_features,
            max_qubits=max_qubits,
            random_state=random_state_i,
        )
        n_qubits = get_n_qubits(X_train_q.shape[1], embedding_type)

        for ansatz_name in ansatz_names:
            print(f"\n=== [Repetition {i + 1}/{n_repeats}] Training VQC "
                  f"(OneVsRest) with ansatz: {ansatz_name} ===")
            _, metrics = train_and_evaluate(
                ansatz_name, X_train_q, X_test_q, y_train, y_test,
                n_qubits=n_qubits, n_layers=n_layers,
                embedding_type=embedding_type, reupload=reupload, n_epochs=n_epochs,
                lr=lr, batch_size=batch_size, random_state=random_state_i,
                measure_all_qubits=measure_all_qubits,
            )
            repetitions_per_model[ansatz_name].append(metrics)

            print(f"Accuracy:            {metrics['accuracy']:.4f}")
            print(f"Precision (macro):   {metrics['precision_macro']:.4f}"
                  f" | (weighted): {metrics['precision_weighted']:.4f}")
            print(f"Recall (macro):      {metrics['recall_macro']:.4f}"
                  f" | (weighted): {metrics['recall_weighted']:.4f}")
            print(f"F1-score (macro):    {metrics['f1_macro']:.4f}"
                  f" | (weighted): {metrics['f1_weighted']:.4f}")

        if classical_names:
            classical_results_i = run_classical_baselines(
                X_train_raw, X_test_raw, y_train, y_test,
                include_svm=include_svm_baseline, svm_kernels=svm_kernels,
                include_decision_tree=include_decision_tree,
                include_mlp=include_mlp,
                include_random_forest=include_random_forest,
                cv_splits=svm_cv_splits, random_state=random_state_i,
                dim_reduction=(
                    dim_reduction
                    if dim_reduction is not None
                    else ("pca" if embedding_type == "angle" and X_train_raw.shape[1] > max_qubits else None)
                ),
                n_final_features=(
                    n_final_features if dim_reduction is not None
                    else (max_qubits if embedding_type == "angle" and X_train_raw.shape[1] > max_qubits else None)
                ),
            )
            for model_name, metrics in classical_results_i.items():
                repetitions_per_model[model_name].append(metrics)

        if callback_checkpoint is not None:
            callback_checkpoint(repetitions_per_model, i)

    # build the final dictionary: flat format (same as before) when
    # n_repeats == 1, or {"repeticoes": [...], "resumo": {...}} when > 1
    results = {}
    for name, metrics_list in repetitions_per_model.items():
        if n_repeats == 1:
            results[name] = metrics_list[0]
        else:
            results[name] = {
                "repeticoes": metrics_list,
                "resumo": aggregate_repetitions(metrics_list),
            }

    if n_repeats > 1:
        print(f"\n{'=' * 80}\nSUMMARY — mean ± standard deviation over {n_repeats} repetitions"
              f"\n{'=' * 80}")
        for name in ansatz_names + classical_names:
            r = results[name]["resumo"]
            print(f"{name:22s} acc={r['accuracy_mean']:.4f}±{r['accuracy_std']:.4f}"
                  f" | f1(weighted)={r['f1_weighted_mean']:.4f}±{r['f1_weighted_std']:.4f}")

    if plot_path is not None:
        plot_results(results, output_path=plot_path, title=title)

    return results


def generate_configuration_key(**configuration_parameters):
    """
    Generates a deterministic, human-readable string key from the experiment
    configuration parameters (e.g., n_final_features, n_layers, n_epochs,
    dim_reduction, embedding_type...).

    It is used as the OUTERMOST level of the results dictionary (see
    run_pipeline / run_mental_health_pipeline), so DIFFERENT configurations
    of the same experiment (e.g., two runs varying n_layers or dim_reduction)
    occupy different positions in the dictionary — and in the pickle — instead
    of overwriting one another.

    Parameter pairs are sorted by parameter name, so the SAME configuration
    always generates the SAME key regardless of the order in which parameters
    were passed. This is the desired behavior: rerunning the SAME configuration
    should update that specific slot (not duplicate it), while different
    configurations receive new slots.

    Example: generate_configuration_key(n_layers=2, n_epochs=60,
    dim_reduction="pca", n_final_features=8) ->
    "n_epochs=60|n_features_final=8|n_layers=2|reducao_dim=pca"
    """
    parts = [f"{name}={configuration_parameters[name]}" for name in sorted(configuration_parameters)]
    return "|".join(parts)


def load_existing_results(output_path):
    """
    Loads the results dictionary already saved at `output_path`, if the file
    exists; otherwise returns an empty dictionary. It is used both for merging
    in save_results_pickle and for the pipelines to check, BEFORE training,
    whether a configuration/disorder has already been executed (see force_rerun
    in run_pipeline / run_mental_health_pipeline).

    Robustness: if the file exists but is empty/corrupted (e.g., a previous
    run was interrupted IN THE MIDDLE of writing the pickle — something that
    could leave a truncated file before atomic writing was implemented in
    save_results_pickle), the exception is NOT propagated. Instead, the
    problematic file is isolated as a backup (suffix ".corrompido") and the
    previous results are treated as empty, with a warning. This prevents a run
    from being interrupted because a previous pickle was not completely saved.
    """
    if os.path.exists(output_path):
        try:
            with open(output_path, "rb") as f:
                return pickle.load(f)
        except (EOFError, pickle.UnpicklingError) as e:
            import shutil
            backup = output_path + ".corrompido"
            shutil.move(output_path, backup)
            print(f"[WARNING] {output_path} was corrupted/empty "
                  f"({type(e).__name__}: {e}). File moved to "
                  f"'{backup}' and previous results were treated as "
                  f"empty. No data is silently lost — "
                  f"the backup remains on disk for inspection.")
            return {}
    return {}


def _merge_two_levels(existing, new):
    """
    Merges `new` into `existing` for up to TWO levels:
      - level 0 (configuration keys; see generate_configuration_key):
        configurations with new keys are simply added;
      - when the SAME configuration key already exists on both sides and
        both values are dictionaries, one additional level is merged
        (disorder names in the mental-health pipeline, or ansatz/SVM names
        in the standard pipeline), preserving entries at that level that were
        not touched in this run and overwriting only entries with the same name.
    It does not merge deeper than this: a retrained disorder/ansatz is replaced
    as a whole rather than merged field by field.
    """
    result = dict(existing)
    for key, new_value in new.items():
        if (key in result
                and isinstance(result[key], dict)
                and isinstance(new_value, dict)):
            merged_subdict = dict(result[key])
            merged_subdict.update(new_value)
            result[key] = merged_subdict
        else:
            result[key] = new_value
    return result


def save_results_pickle(results, output_path="/mnt/user-data/outputs/resultados_vqc.pkl",
                              merge_with_existing=False):
    """
    Saves the complete results dictionary (metrics + trained parameters from
    all VQC ansätze/circuits and, optionally, SVM baselines) to a pickle file.

    merge_with_existing : bool
        If False (default), completely OVERWRITES any pickle already existing
        at `output_path`. Calling this function more than once with the same
        path DISCARDS previously saved results.
        If True, loads the existing pickle at `output_path` (if present) and
        merges it with `results` for up to two levels (see
        _merge_two_levels): a NEW configuration key is simply added; if the
        SAME configuration key already exists, lower-level entries (disorders
        in the mental-health pipeline; ansätze/SVM in the standard pipeline)
        are merged, replacing only entries with the SAME NAME while preserving
        all others.
        In run_pipeline / run_mental_health_pipeline, the top-level key is now
        the experiment CONFIGURATION KEY (see generate_configuration_key).
        Therefore, with merge_with_existing=True (the default in both
        pipelines), running the same script multiple times with different
        n_final_features/n_layers/n_epochs/dim_reduction values ACCUMULATES
        each configuration in its own slot in the same pickle without deleting
        previous configurations. Rerunning the SAME configuration only updates
        (merges) that specific slot.

    Structure of the saved dictionary (from run_pipeline /
    run_mental_health_pipeline):
        {
            "<configuration_key_1>": {              # see generate_configuration_key
                # single-dataset format (run_pipeline):
                "<ansatz_name>": {...}, "svm_<kernel>": {...}, ...,
                "_metadata": {...},
                # OR format nested by disorder (run_mental_health_pipeline):
                "<main_disorder_1>": {"<ansatz_name>": {...}, ...},
                "<main_disorder_2>": {...},
                "_metadata": {...},
            },
            "<configuration_key_2>": {...},         # another config, another slot
            ...
        }

    Each model entry ("<ansatz_name>" or "svm_<kernel>") has the format:
        {
            "accuracy": float,
            "precision_macro": float, "precision_weighted": float,
            "recall_macro": float, "recall_weighted": float,
            "f1_macro": float, "f1_weighted": float,
            "classification_report": dict (per-class metrics),
            "confusion_matrix": np.ndarray,
            "y_true": np.ndarray, "y_pred": np.ndarray,
            "params": {...trained weights/parameters...},
            "config": {...hyperparameters used...},
        }
    (SVM also has "best_params" and "model"; see train_and_evaluate /
    train_evaluate_svm.)

    If the dictionary was generated with n_repeats > 1 (see
    train_evaluate_all_models), each model entry above instead has the format:
        {
            "repeticoes": [metrics from repetition 0, metrics from repetition 1, ...],
            "resumo": {
                "n_repeats": int,
                "accuracy_mean": float, "accuracy_std": float,
                "precision_macro_mean": float, "precision_macro_std": float,
                ... (same _mean/_std pattern for all scalar metrics)
            },
        }
    """
    final_data = results

    if merge_with_existing and os.path.exists(output_path):
        existing_data = load_existing_results(output_path)
        n_before = len(existing_data)
        final_data = _merge_two_levels(existing_data, results)
        print(f"Merging with existing pickle ({n_before} configuration(s)) — "
              f"final result with {len(final_data)} configuration(s).")

    # ATOMIC write: first write to a temporary file and only replace
    # the final file via os.replace() after pickle.dump completes
    # successfully. os.replace() is an atomic operation on the same filesystem —
    # the final file contains the COMPLETE new content, or (if the process
    # is interrupted midway) retains the PREVIOUS content intact.
    # The final file is NEVER left truncated/corrupted in the middle of a write,
    # which was the root cause of EOFError in load_existing_results.
    tmp_path = output_path + ".tmp"
    with open(tmp_path, "wb") as f:
        pickle.dump(final_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp_path, output_path)
    print(f"Results (metrics + parameters) saved to: {output_path}")


# ==============================================================================
# 7. MAIN PIPELINE
# ==============================================================================

def run_pipeline(dataset_name="iris", embedding_type="angle",
                       dim_reduction=None, n_final_features=None,
                       n_layers=2, reupload=True, n_epochs=60, lr=0.1,
                       batch_size=None, max_qubits=4, test_size=0.3,
                       random_state=42, n_repeats=1,
                       include_svm_baseline=True,
                       svm_kernels=("linear", "rbf", "poly", "sigmoid"),
                       svm_cv_splits=5,
                       include_decision_tree=True,
                       include_mlp=True,
                       include_random_forest=True,
                       pickle_path="/mnt/user-data/outputs/resultados_vqc.pkl",
                       merge_with_existing=True, force_rerun=False,
                       measure_all_qubits=False):
    """
    Runs the complete pipeline:
      load data (with optional PCA/UMAP dimensionality reduction)
      -> check whether this configuration has already been executed (see
      force_rerun; if it has, SKIP training and return the saved result)
      -> split into training/test sets -> train the five ansätze (each inside
      a OneVsRestClassifier, with data re-uploading and parameters optimized
      using JAX + Optax) -> [optional] train classical baselines (SVM,
      Decision Tree, MLP, and Random Forest, each through GridSearchCV) on
      the SAME training/test set -> evaluate (accuracy, precision, recall,
      f1, confusion matrix) -> plot -> save metrics + trained parameters
      (VQCs and classical models) in a single pickle. If n_repeats > 1,
      the entire process above is repeated n_repeats times using different
      train/test splits (see train_evaluate_all_models).

    Parameters
    ----------
    dataset_name : "iris" or "wine"
    embedding_type : "angle" (phase/angle embedding) or "amplitude"
    dim_reduction : None, "pca", or "umap"
        Dimensionality-reduction method applied to features before quantum
        embedding. If None, use the default behavior (see load_data:
        automatic PCA fallback only for "angle" embedding when the number
        of features exceeds max_qubits).
    n_final_features : int or None
        Desired final number of features after reduction. Required when
        dim_reduction is "pca" or "umap"; in practice, it also defines the
        number of qubits when embedding_type="angle" (1 feature = 1 qubit).
    n_layers : number of variational layers in each ansatz
    reupload : bool
        If True (default), each of the n_layers layers is composed of
        [data loading + ansatz block] — the data are re-uploaded at each
        layer ("data re-uploading"), increasing circuit expressivity.
        If False, data are loaded only once before the first layer.
    n_epochs : training epochs per binary classifier
    lr : learning rate for the Adam optimizer (optax.adam)
    batch_size : mini-batch size; None = full-batch (all training data at
                 each step, more stable and common for small VQCs)
    max_qubits : maximum number of qubits allowed when embedding_type="angle"
                 and dim_reduction=None (automatic PCA fallback)
    random_state : base seed; repetition i uses random_state + i
    n_repeats : int
        Number of experiment repetitions, each with a different train/test
        split (random_state + i). In each repetition, the SAME split is used
        to train both the VQCs and all classical baselines. If 1 (default),
        behavior is equivalent to a single run; if >1, each result entry
        contains {"repeticoes": [...], "resumo": {means and standard deviations}}.
    include_svm_baseline : bool
        If True (default), also train/evaluate an SVM (sklearn) with
        GridSearchCV for each kernel in `svm_kernels`, using exactly the same
        X_train/X_test/y_train/y_test as the VQCs in each repetition, and add
        the results (keys "svm_<kernel>") to the final dictionary and pickle.
    svm_kernels : SVM kernels to test through grid search
    svm_cv_splits : number of StratifiedKFold folds used in GridSearchCV
        (shared by SVM, Decision Tree, MLP, and Random Forest)
    include_decision_tree : bool; if True (default), include a Decision Tree
        (sklearn.tree.DecisionTreeClassifier) with GridSearchCV as a baseline
    include_mlp : bool; if True (default), include an MLP neural network
        (sklearn.neural_network.MLPClassifier) with GridSearchCV. The grid
        covers several architectures/regularization settings with early
        stopping to obtain a well-tuned configuration rather than an
        arbitrary fixed architecture.
    include_random_forest : bool; if True (default), include a Random Forest
        (sklearn.ensemble.RandomForestClassifier) with GridSearchCV, suggested
        as an additional baseline because it is typically strong and robust
        on tabular data with little tuning required.
    pickle_path : path where the results dictionary will be saved
    merge_with_existing : bool
        If True (default), results from this run are saved under their own
        CONFIGURATION KEY (generated from dataset_name, embedding_type,
        dim_reduction, n_final_features, n_layers, n_epochs, reupload, and
        n_repeats; see generate_configuration_key) and merged with any pickle
        already present at `pickle_path`. DIFFERENT configurations occupy
        different slots in the same dictionary/pickle without overwriting one
        another; rerunning the SAME configuration only updates that specific
        slot. If False, each call completely overwrites the pickle at
        `pickle_path`, even if it contains only this run's configuration.
    force_rerun : bool
        If False (default) and merge_with_existing=True, check BEFORE training
        whether this run's configuration key already exists in the pickle at
        `pickle_path`. If it does, the entire training procedure is SKIPPED
        and the function directly returns the saved result (no additional
        quantum-simulation or GridSearchCV time is spent). If True, train and
        overwrite that slot even if it already exists.
    """
    print(f"Dataset: {dataset_name} | Embedding: {embedding_type} | "
          f"Dim. reduction: {dim_reduction if dim_reduction else 'none (automatic fallback if required)'} | "
          f"Layers: {n_layers} | Data re-uploading: {reupload} | Epochs: {n_epochs} | "
          f"Repetitions: {n_repeats}")

    X, y, target_names, reducer = load_data(
        dataset_name, embedding_type,
        dim_reduction=dim_reduction, n_final_features=n_final_features,
        max_qubits=max_qubits, random_state=random_state,
    )
    n_planned_features = (
        n_final_features if dim_reduction is not None
        else (max_qubits if embedding_type == "angle" and X.shape[1] > max_qubits else X.shape[1])
    )
    n_qubits = get_n_qubits(n_planned_features, embedding_type)
    print(f"No. of original features: {X.shape[1]} | "
          f"No. of planned features after reduction: {n_planned_features} | "
          f"No. of qubits used: {n_qubits} | Classes: {list(target_names)}")

    # Configuration key: ensures that rerunning this pipeline with
    # n_features_final/n_layers/n_epochs/reducao_dim (ou embedding_type,
    # reupload, dataset, n_repeats) values occupies its OWN SLOT in the
    # dictionary/pickle (and in separate plot files), instead of
    # overwriting the result of another configuration.
    config_key = generate_configuration_key(
        dataset=dataset_name,
        embedding_type=embedding_type,
        dim_reduction=dim_reduction,
        n_final_features=n_planned_features,
        n_layers=n_layers,
        n_epochs=n_epochs,
        reupload=reupload,
        n_repeats=n_repeats,
        measure_all_qubits=measure_all_qubits,
        preprocessing_version="split_aware_v2",
    )
    config_suffix = config_key.replace("|", "_").replace("=", "-")
    print(f"Configuration key for this experiment: {config_key}")

    # Skip training if this configuration has already been executed
    # (it is already saved in the pickle) and force_rerun=False. This only makes sense
    # when merge_with_existing=True, because this is the mode that
    # accumulates configurations in the same pickle.
    if merge_with_existing and not force_rerun:
        existing = load_existing_results(pickle_path)
        if config_key in existing:
            print(f"\nConfiguration already executed previously — skipping training "
                  f"(use force_rerun=True to train again anyway).")
            return {config_key: existing[config_key]}

    results = train_evaluate_all_models(
        X, y, target_names,
        n_layers=n_layers, reupload=reupload, n_epochs=n_epochs, lr=lr,
        batch_size=batch_size, embedding_type=embedding_type,
        dim_reduction=dim_reduction, n_final_features=n_final_features,
        max_qubits=max_qubits,
        test_size=test_size, random_state=random_state, n_repeats=n_repeats,
        include_svm_baseline=include_svm_baseline, svm_kernels=svm_kernels,
        svm_cv_splits=svm_cv_splits,
        include_decision_tree=include_decision_tree,
        include_mlp=include_mlp,
        include_random_forest=include_random_forest,
        plot_path=f"/mnt/user-data/outputs/comparacao_ansatze__{config_suffix}.png",
        title=f"Comparison of Ansätze (VQC) vs. Classical Baselines — {dataset_name}",
        measure_all_qubits=measure_all_qubits,
    )

    # general execution metadata, together with results for each ansatz
    results["_metadata"] = {
        "timestamp": datetime.now().isoformat(),
        "dataset": dataset_name,
        "embedding_type": embedding_type,
        "reducao_dim": dim_reduction,
        "n_features_final": n_planned_features,
        "reducer_usado": dim_reduction,
        "n_qubits": n_qubits,
        "n_layers": n_layers,
        "reupload": reupload,
        "measure_all_qubits": measure_all_qubits,
        "classes": list(target_names),
        "ansätze_avaliados": list(ANSATZ_REGISTRY.keys()),
        "svm_baseline_incluido": include_svm_baseline,
        "svm_kernels_avaliados": list(svm_kernels) if include_svm_baseline else [],
        "decision_tree_incluido": include_decision_tree,
        "mlp_incluido": include_mlp,
        "random_forest_incluido": include_random_forest,
        "n_repeats": n_repeats,
        "random_state_base": random_state,
        "preprocessing_version": "split_aware_v2",
    }

    final_results = {config_key: results}

    save_results_pickle(final_results, pickle_path, merge_with_existing=merge_with_existing)
    return final_results


# ==============================================================================
# 8. PIPELINE FOR THE MENTAL-HEALTH DATASET (multi-disorder)
# ==============================================================================

def run_mental_health_pipeline(csv_path,
                                    disorders=None,
                                    column_main="main.disorder",
                                    column_specific="specific.disorder",
                                    group_control="Healthy control",
                                    columns_exclude=("sex_M",),
                                    embedding_type="angle",
                                    dim_reduction="pca",
                                    n_final_features=8,
                                    max_qubits=8,
                                    n_layers=2, reupload=True, n_epochs=60,
                                    lr=0.1, batch_size=None, test_size=0.3,
                                    random_state=42, n_repeats=1,
                                    include_svm_baseline=True,
                                    svm_kernels=("linear", "rbf", "poly", "sigmoid"),
                                    svm_cv_splits=5,
                                    include_decision_tree=True,
                                    include_mlp=True,
                                    include_random_forest=True,
                                    output_dir="/mnt/user-data/outputs",
                                    pickle_prefix="resultados_saude_mental",
                                    merge_with_existing=True,
                                    force_rerun=False,
                                    measure_all_qubits=False,
                                    checkpoint_per_repetition=True):
    """
    Same behavior and function signature as the original version (see the
    previous docstring), with ONE important behavioral change:

    Previously, the pickle was written to disk only after ALL disorders
    (and, within each one, ALL `n_repeats` repetitions) had finished.
    Now:
      - after EACH completed repetition within a disorder, a partial
        checkpoint for that disorder is saved to disk (unless
        `checkpoint_per_repetition=False`);
      - at the end of EACH disorder, another checkpoint is saved, now
        complete and containing final metadata.

    RESUMPTION: when the pipeline is (re)started, if a disorder is already
    present in the pickle under the SAME configuration key but contains
    FEWER than `n_repeats` saved repetitions, training resumes from the next
    repetition instead of restarting from scratch. No previously completed
    training time is wasted.

    checkpoint_per_repetition : bool
        If True (default), saves a checkpoint after every completed repetition
        (fine-grained persistence — recommended when n_epochs/n_repeats are
        large, as in runs with many measurements). If False, saves only at the
        end of each disorder (closer to the previous behavior, but still
        without waiting for ALL disorders to finish).

    PICKLE FILE NAME
    ----------------
    The file name now EMBEDS the overall experiment configuration
    (the same variables used in `config_key`/`generate_configuration_key`:
    embedding_type, dim_reduction, n_final_features, n_layers, n_epochs,
    reupload, n_repeats, and measure_all_qubits), instead of using a fixed name:

        <output_dir>/<pickle_prefix>__<config_suffix>.pkl

    Example: "mental_health_results__embedding_type-angle_measure_all_qubits-True_"
             "n_epochs-300_n_features_final-6_n_layers-10_n_repeats-10_"
             "reducao_dim-umap_reupload-True.pkl"

    Rationale: each hyperparameter combination now lives in ITS OWN file
    instead of accumulating all configurations inside one large generic
    pickle. The file name immediately shows WHICH configuration produced a
    result, without opening the pickle and inspecting `config_key`, and runs
    with different configurations do not compete to write to the same file.
    The INTERNAL pickle structure is unchanged:
    `{config_key: {<disorder>: {...}, "_metadata": {...}}}`. Each file now
    normally contains a single `config_key` (the one corresponding to its
    own file name), unless files are manually merged later.

    pickle_prefix : str
        Prefix placed before the configuration in the file name (see above).
        Use it to distinguish experiment families that you want to keep in
        separate groups of files (e.g., different datasets), even when they
        share the same `config_key`.
    """
    df = load_mental_health_csv(csv_path)

    if disorders is None:
        disorders = list_main_disorders(df, column_main, group_control)
    print(f"\nDesordens principais a treinar: {disorders}")

    config_key = generate_configuration_key(
        embedding_type=embedding_type,
        dim_reduction=dim_reduction,
        n_final_features=n_final_features,
        n_layers=n_layers,
        n_epochs=n_epochs,
        reupload=reupload,
        n_repeats=n_repeats,
        measure_all_qubits=measure_all_qubits,
        preprocessing_version="split_aware_v2",
    )
    config_suffix = config_key.replace("|", "_").replace("=", "-")
    print(f"Configuration key for this experiment: {config_key}")

    os.makedirs(output_dir, exist_ok=True)
    # the file name INCLUDES the overall experiment configuration — see
    # docstring above ("PICKLE FILE NAME") for the complete format
    pickle_path = f"{output_dir}/{pickle_prefix}__{config_suffix}.pkl"
    print(f"Results file for this configuration: {pickle_path}")

    already_trained_disorders = {}
    if merge_with_existing and not force_rerun:
        existing = load_existing_results(pickle_path)
        config_content = existing.get(config_key, {})
        already_trained_disorders = {
            k: v for k, v in config_content.items() if k != "_metadata"
        }
        if already_trained_disorders:
            print(f"Disorders already present in this configuration (they may be "
                  f"complete or partial): {list(already_trained_disorders.keys())}")

    all_model_names = _all_model_names(
        include_svm_baseline, svm_kernels, include_decision_tree,
        include_mlp, include_random_forest,
    )

    complete_results = {}

    def _save_checkpoint(overall_complete=False):
        """Persists the CURRENT state of `complete_results` to the pickle,
        always merging it (intermediate checkpoints never overwrite
        other already-saved configurations/disorders)."""
        complete_results["_metadata"] = {
            "timestamp": datetime.now().isoformat(),
            "csv": csv_path,
            "embedding_type": embedding_type,
            "reducao_dim": dim_reduction,
            "n_features_final": n_final_features,
            "n_layers": n_layers,
            "reupload": reupload,
            "measure_all_qubits": measure_all_qubits,
            "desordens_alvo": list(disorders),
            "desordens_concluidas_neste_checkpoint": [
                d for d in complete_results if d != "_metadata"
            ],
            "svm_baseline_incluido": include_svm_baseline,
            "decision_tree_incluido": include_decision_tree,
            "mlp_incluido": include_mlp,
            "random_forest_incluido": include_random_forest,
            "n_repeats": n_repeats,
            "random_state_base": random_state,
            "preprocessing_version": "split_aware_v2",
            "pipeline_completo": overall_complete,
        }
        final_results = {config_key: dict(complete_results)}
        save_results_pickle(final_results, pickle_path,
                                  merge_with_existing=True)

    for disorder in disorders:
        already_saved = already_trained_disorders.get(disorder)
        existing_repetitions = (
            _extract_partial_repetitions(already_saved, all_model_names)
            if already_saved is not None else None
        )

        if n_repeats == 1:
            # Without repetitions: it either already exists (flat format) or does not exist.
            if already_saved is not None:
                print(f"\n=== Disorder '{disorder}' already trained in this "
                      f"configuration — skipping ===")
                complete_results[disorder] = already_saved
                _save_checkpoint()
                continue
        else:
            n_already_done = (
                len(next(iter(existing_repetitions.values())))
                if existing_repetitions else 0
            )
            if n_already_done >= n_repeats:
                print(f"\n=== Disorder '{disorder}' already has all {n_repeats} "
                      f"repetitions completed in this configuration — skipping ===")
                complete_results[disorder] = already_saved
                _save_checkpoint()
                continue
            elif n_already_done > 0:
                print(f"\n=== Disorder '{disorder}': resuming from where it stopped "
                      f"({n_already_done}/{n_repeats} repetitions already saved) ===")

        print(f"\n{'=' * 80}\nMAIN DISORDER: {disorder}\n{'=' * 80}")

        X, y, class_names = prepare_disorder_data(
            df, disorder,
            column_main=column_main,
            column_specific=column_specific,
            group_control=group_control,
            columns_exclude=columns_exclude,
        )

        n_planned_features = (
            n_final_features if dim_reduction is not None
            else (max_qubits if embedding_type == "angle" and X.shape[1] > max_qubits else X.shape[1])
        )
        n_qubits = get_n_qubits(n_planned_features, embedding_type)
        reducer = None
        print(f"No. of original features: {X.shape[1]} | "
              f"No. of planned features after reduction: {n_planned_features} | "
              f"No. of qubits used: {n_qubits} | Classes: {class_names}")

        file_name = disorder.lower().replace(" ", "_").replace("/", "-")

        def _repetition_checkpoint_callback(repetitions_per_model, completed_index,
                                            disorder=disorder, class_names=class_names,
                                            n_qubits=n_qubits, X=X, reducer=reducer):
            partial = {}
            for name, items_list in repetitions_per_model.items():
                if n_repeats == 1:
                    partial[name] = items_list[0] if items_list else None
                else:
                    partial[name] = {
                        "repeticoes": list(items_list),
                        "resumo": aggregate_repetitions(items_list),
                    }
            partial["_metadata"] = {
                "desordem_principal": disorder,
                "classes": class_names,
                "n_qubits": n_qubits,
                "n_features_final": n_planned_features,
                "reducer_usado": dim_reduction,
                "n_repeats": n_repeats,
                "n_repeats_concluidos": completed_index + 1,
                "completo": (completed_index + 1) == n_repeats,
            }
            complete_results[disorder] = partial
            _save_checkpoint()
            print(f"[checkpoint] Disorder '{disorder}': repetition "
                  f"{completed_index + 1}/{n_repeats} saved to disk "
                  f"({pickle_path}).")

        disorder_results = train_evaluate_all_models(
            X, y, class_names,
            n_layers=n_layers, reupload=reupload, n_epochs=n_epochs, lr=lr,
            batch_size=batch_size, embedding_type=embedding_type,
            dim_reduction=dim_reduction, n_final_features=n_final_features,
            max_qubits=max_qubits,
            test_size=test_size, random_state=random_state, n_repeats=n_repeats,
            include_svm_baseline=include_svm_baseline, svm_kernels=svm_kernels,
            svm_cv_splits=svm_cv_splits,
            include_decision_tree=include_decision_tree,
            include_mlp=include_mlp,
            include_random_forest=include_random_forest,
            plot_path=f"{output_dir}/comparacao_{file_name}__{config_suffix}.png",
            title=f"Comparison of Ansätze (VQC) vs. Classical Baselines — {disorder}",
            measure_all_qubits=measure_all_qubits,
            existing_repetitions=existing_repetitions,
            callback_checkpoint=(_repetition_checkpoint_callback
                                  if checkpoint_per_repetition else None),
        )

        disorder_results["_metadata"] = {
            "desordem_principal": disorder,
            "classes": class_names,
            "n_qubits": n_qubits,
            "n_features_final": n_planned_features,
            "reducer_usado": dim_reduction,
            "n_repeats": n_repeats,
            "n_repeats_concluidos": n_repeats,
            "completo": True,
        }
        complete_results[disorder] = disorder_results

        # END-OF-DISORDER checkpoint (ensures consistent metadata
        # even when checkpoint_per_repetition=False).
        _save_checkpoint()
        print(f"[checkpoint] Disorder '{disorder}' completed and saved to disk.")

    # Final checkpoint: marks the pipeline as complete for this call.
    _save_checkpoint(overall_complete=True)

    final_results = {config_key: complete_results}
    if not merge_with_existing:
        # The only case in which behavior differs from a regular checkpoint:
        # the user explicitly requested NOT to merge — overwrite the
        # entire pickle with only this configuration key.
        save_results_pickle(final_results, pickle_path,
                                  merge_with_existing=False)
    return final_results


# ==============================================================================
# 9. RESULTS COMPARISON TABLE
# ==============================================================================

def _is_model_result(d):
    """Heuristic: a dictionary is treated as a model result if it contains
    single-run metric keys ("accuracy") or the aggregated repetition format
    ("resumo")."""
    return isinstance(d, dict) and ("accuracy" in d or "resumo" in d)


def _model_type(model_name):
    """
    Classifies a model name into a human-readable type for the comparison
    table and plot coloring:
      - any name present in ANSATZ_REGISTRY -> "VQC"
      - "svm_<kernel>" -> "SVM"
      - "decision_tree" -> "Decision Tree"
      - "mlp" -> "MLP"
      - "random_forest" -> "Random Forest"
      - any other name -> "Other" (fallback for future extensions)
    """
    if model_name in ANSATZ_REGISTRY:
        return "VQC"
    if model_name.startswith("svm_"):
        return "SVM"
    known_names = {
        "decision_tree": "Decision Tree",
        "mlp": "MLP",
        "random_forest": "Random Forest",
    }
    return known_names.get(model_name, "Other")


def _extract_model_row(model_name, model_data):
    """
    Flattens a model entry in `results` (either a single-run format or the
    n_repeats>1 aggregate) into a one-row dictionary ready to become a
    DataFrame row.
    """
    type = _model_type(model_name)

    if "resumo" in model_data:
        r = model_data["resumo"]
        row = {
            "accuracy": r["accuracy_mean"], "accuracy_std": r["accuracy_std"],
            "precision_macro": r["precision_macro_mean"],
            "precision_macro_std": r["precision_macro_std"],
            "precision_weighted": r["precision_weighted_mean"],
            "precision_weighted_std": r["precision_weighted_std"],
            "recall_macro": r["recall_macro_mean"],
            "recall_macro_std": r["recall_macro_std"],
            "recall_weighted": r["recall_weighted_mean"],
            "recall_weighted_std": r["recall_weighted_std"],
            "f1_macro": r["f1_macro_mean"], "f1_macro_std": r["f1_macro_std"],
            "f1_weighted": r["f1_weighted_mean"], "f1_weighted_std": r["f1_weighted_std"],
            "n_repeats": r["n_repeats"],
        }
        reference = model_data["repeticoes"][0]  # config is the same in all repetitions
    else:
        row = {
            "accuracy": model_data["accuracy"], "accuracy_std": 0.0,
            "precision_macro": model_data["precision_macro"], "precision_macro_std": 0.0,
            "precision_weighted": model_data["precision_weighted"], "precision_weighted_std": 0.0,
            "recall_macro": model_data["recall_macro"], "recall_macro_std": 0.0,
            "recall_weighted": model_data["recall_weighted"], "recall_weighted_std": 0.0,
            "f1_macro": model_data["f1_macro"], "f1_macro_std": 0.0,
            "f1_weighted": model_data["f1_weighted"], "f1_weighted_std": 0.0,
            "n_repeats": 1,
        }
        reference = model_data

    row["modelo"] = model_name
    row["tipo"] = type

    config = reference.get("config", {})
    if type == "VQC":
        row["configuracao"] = (
            f"ansatz={config.get('ansatz', model_name)}, "
            f"n_layers={config.get('n_layers')}, "
            f"embedding={config.get('embedding_type')}, "
            f"reupload={config.get('reupload')}, "
            f"lr={config.get('lr')}, epochs={config.get('n_epochs')}"
        )
    elif type == "SVM":
        row["configuracao"] = f"kernel={config.get('kernel')}, best_params={config.get('best_params')}"
    else:
        # Decision Tree, MLP, Random Forest (ou outro futuro baseline
        # classical): best_params already contains the winning hyperparameters from
        # that model's GridSearchCV
        row["configuracao"] = f"best_params={config.get('best_params')}"

    return row


def _group_models_by_group(results):
    """
    Traverses `results` (using the same formats accepted by
    generate_comparison_table: with or without a configuration key and with
    or without nesting by main disorder) and returns a flat dictionary:
        {(experiment, dataset): {model_name: model_data, ...}, ...}

    Centralizes the format-detection logic used by both
    generate_comparison_table and generate_statistical_tests_table so that
    both functions always operate on the same groups of comparable models
    (same configuration + same dataset).
    """
    groups = {}

    for key, content in results.items():
        if key == "_metadata" or not isinstance(content, dict):
            continue

        if _is_model_result(content):
            # old format (without a configuration key): `key` is already the
            # model name itself, and `results` is a flat dictionary for one dataset
            name_dataset = results.get("_metadata", {}).get("dataset", "dataset")
            group = groups.setdefault(("(without configuration key)", name_dataset), {})
            group[key] = content
            continue

        subkeys = [k for k in content.keys() if k != "_metadata"]
        if not subkeys:
            continue

        # `key` is a configuration key; determine whether `content` is the
        # flat format for one dataset or nested by dataset/disorder
        if _is_model_result(content[subkeys[0]]):
            name_dataset = content.get("_metadata", {}).get("dataset", "dataset")
            dataset_groups = {name_dataset: {k: content[k] for k in subkeys}}
        else:
            dataset_groups = {k: content[k] for k in subkeys}

        for name_dataset, models in dataset_groups.items():
            if not isinstance(models, dict):
                continue
            group = groups.setdefault((key, name_dataset), {})
            for model_name, model_data in models.items():
                if model_name == "_metadata":
                    continue
                group[model_name] = model_data

    return groups


def generate_comparison_table(results, sorting_metric="f1_weighted",
                              only_best=True, csv_path=None):
    """
    Generates a comparison table (pandas.DataFrame) containing the best
    configuration for each MODEL TYPE — VQC (quantum) and, among the
    classical models, SVM (best kernel), Decision Tree, MLP, and Random
    Forest — for each (configuration key, dataset) combination. See
    _model_type for the classification used.

    It directly accepts the dictionary returned by the current versions of
    run_pipeline or run_mental_health_pipeline, whose outermost key is the
    experiment CONFIGURATION KEY (see generate_configuration_key). This is
    useful when the same pickle accumulates several different configurations
    (n_layers, n_epochs, dim_reduction, etc.; see merge_with_existing).
    Within each configuration, it accepts either the ONE-dataset format
    (keys = model names — ansatz name, "svm_<kernel>", "decision_tree",
    "mlp", or "random_forest") or the format nested by main disorder
    (keys = disorder names, each with its own model dictionary). For
    compatibility with pickle files saved by previous versions (without a
    configuration key), it also accepts the raw dictionary in either form.
    All formats are detected automatically. It works with n_repeats=1 or >1;
    in the latter case, the mean of each metric is used (see
    train_evaluate_all_models).

    Parameters
    ----------
    results : dict returned by run_pipeline or run_mental_health_pipeline
    sorting_metric : metric used to choose the best configuration of each type
        (default: "f1_weighted"; accepts any scalar metric present, e.g.,
        "accuracy", "recall_macro", etc.)
    only_best : bool
        If True (default), keep only the BEST row of each type (VQC, SVM,
        Decision Tree, MLP, Random Forest) for each (experiment, dataset)
        according to `sorting_metric` — this is the comparison table itself.
        If False, return ALL trained models in each combination without filtering.
    csv_path : if provided, also save the table to this CSV path.

    Returns
    -------
    pandas.DataFrame with columns:
        experiment, dataset, type, model,
        accuracy, accuracy_std,
        precision_macro, precision_macro_std,
        precision_weighted, precision_weighted_std,
        recall_macro, recall_macro_std,
        recall_weighted, recall_weighted_std,
        f1_macro, f1_macro_std, f1_weighted, f1_weighted_std,
        n_repeats, configuration
    The "*_std" fields come from the standard deviation across repetitions
    (n_repeats>1; see aggregate_repetitions) and are 0.0 for a single-run
    result (n_repeats=1).
    """
    groups = _group_models_by_group(results)

    rows = []
    for (experiment, dataset), models in groups.items():
        for model_name, model_data in models.items():
            row = _extract_model_row(model_name, model_data)
            row["base_dados"] = dataset
            row["experimento"] = experiment
            rows.append(row)

    if not rows:
        raise ValueError("No model result could be found in `results`.")

    df = pd.DataFrame(rows)
    columns = ["experimento", "base_dados", "tipo", "modelo",
               "accuracy", "accuracy_std",
               "precision_macro", "precision_macro_std",
               "precision_weighted", "precision_weighted_std",
               "recall_macro", "recall_macro_std",
               "recall_weighted", "recall_weighted_std",
               "f1_macro", "f1_macro_std", "f1_weighted", "f1_weighted_std",
               "n_repeats", "configuracao"]
    df = df[columns]

    if only_best:
        df = (df.sort_values(sorting_metric, ascending=False)
                .groupby(["experimento", "base_dados", "tipo"], as_index=False)
                .first())

    df = df.sort_values(["base_dados", "experimento", "tipo"]).reset_index(drop=True)

    if csv_path is not None:
        df.to_csv(csv_path, index=False)
        print(f"Comparison table saved to: {csv_path}")

    return df


def _extract_repetition_values(model_data, metric):
    """
    Extracts the array containing the value of `metric` for EACH repetition
    of a model (one value per element of model_data["repeticoes"], in the same
    order in which they were trained; see train_evaluate_all_models). Returns
    None if the model is not in the aggregated repetition format (n_repeats == 1).
    """
    if "repeticoes" not in model_data:
        return None
    return np.array([rep[metric] for rep in model_data["repeticoes"]], dtype=float)


def generate_statistical_tests_table(results, metric="f1_weighted",
                                      test_name="wilcoxon", alpha=0.05,
                                      csv_path=None):
    """
    Generates a table of PAIRED statistical tests comparing ALL models
    against one another (all possible pairs: VQC vs VQC, VQC vs classical,
    classical vs classical) within each (experiment, dataset) combination.

    Why paired
    ----------
    Within the same call to run_pipeline / run_mental_health_pipeline with
    n_repeats > 1, ALL models (VQCs and classical models) are evaluated on
    the SAME set of train/test splits in each repetition. Repetition i of
    ANY model uses exactly the same random_state_i and therefore the same
    split (see train_evaluate_all_models). As a result, the metric sequences
    from any two models across repetitions are PAIRED sample by sample
    (rather than being two independent samples), which is exactly the
    assumption required by tests such as Wilcoxon signed-rank or the paired
    t-test. An independent-samples test would be inappropriate here.

    Requires n_repeats > 1 (see train_evaluate_all_models): without repeated
    runs there is no variation across paired executions to test.

    Parameters
    ----------
    results : dict returned by run_pipeline or run_mental_health_pipeline,
              with n_repeats > 1
    metric : metric compared across models (default "f1_weighted"; any scalar
             metric present in each repetition, e.g., "accuracy")
    test_name : "wilcoxon" (default) — Wilcoxon signed-rank test,
        nonparametric and recommended when there are few repetitions or when
        normality of the paired differences cannot be assumed; or
        "ttest" — paired t-test (scipy.stats.ttest_rel), parametric and
        assuming the paired differences are approximately normally distributed
    alpha : significance level used for the "significativo" column
        (p_value < alpha); default 0.05
    csv_path : if provided, also save the table to this CSV path

    Returns
    -------
    pandas.DataFrame with one row per DISTINCT model pair (without repeating
    the reversed pair), for each (experiment, dataset) containing at least
    two models with multiple repetitions, with columns:
        experiment, dataset, model_a, model_b, metric, n_repeats,
        mean_a, mean_b, mean_difference, test, statistic, p_value,
        significant, best
    "significant" is True when p_value < alpha (rejecting the null hypothesis
    that both models have the same performance on that metric). "best"
    identifies the model with the higher mean WHEN the difference is
    statistically significant; otherwise it displays
    "tie (no significant difference)", meaning that neither model can be
    claimed to be better than the other.
    """
    if test_name not in ("wilcoxon", "ttest"):
        raise ValueError("test_name must be 'wilcoxon' or 'ttest'")

    groups = _group_models_by_group(results)

    rows = []
    for (experiment, dataset), models in groups.items():
        # only models with multiple repetitions enter the comparison
        # (this is what provides the paired sample for the statistical test)
        values_per_model = {}
        for model_name, model_data in models.items():
            values = _extract_repetition_values(model_data, metric)
            if values is not None and len(values) > 1:
                values_per_model[model_name] = values

        names = sorted(values_per_model.keys())
        if len(names) < 2:
            continue  # nothing to compare for this (configuration, dataset) combination

        for model_a, model_b in itertools.combinations(names, 2):
            values_a = values_per_model[model_a]
            values_b = values_per_model[model_b]
            if len(values_a) != len(values_b):
                # should not occur (the same n_repeats is used for all models
                # within a single call), but skip the pair as a safeguard
                continue

            diffs = values_a - values_b
            if np.allclose(diffs, 0):
                # no difference in any repetition -> the paired test
                # has nothing to detect; explicitly report a tie
                statistic, p_value = 0.0, 1.0
            else:
                try:
                    if test_name == "wilcoxon":
                        statistic, p_value = stats.wilcoxon(values_a, values_b)
                    else:
                        statistic, p_value = stats.ttest_rel(values_a, values_b)
                except ValueError:
                    # e.g., Wilcoxon with all differences equal to zero after
                    # the default handling of ties
                    statistic, p_value = 0.0, 1.0

            mean_a, mean_b = float(values_a.mean()), float(values_b.mean())
            significant = bool(p_value < alpha)
            if significant:
                best = model_a if mean_a > mean_b else model_b
            else:
                best = "tie (no significant difference)"

            rows.append({
                "experimento": experiment,
                "base_dados": dataset,
                "modelo_a": model_a,
                "modelo_b": model_b,
                "metrica": metric,
                "n_repeats": len(values_a),
                "media_a": mean_a,
                "media_b": mean_b,
                "diferenca_media": mean_a - mean_b,
                "teste": test_name,
                "estatistica": float(statistic),
                "p_valor": float(p_value),
                "significativo": significant,
                "melhor": best,
            })

    if not rows:
        raise ValueError(
            "No statistical test could be generated: the results "
            "must have been generated with n_repeats > 1 (see "
            "train_evaluate_all_models) and contain at least two models per "
            "combination of (experiment, dataset)."
        )

    df = pd.DataFrame(rows)
    df = df.sort_values(["base_dados", "experimento", "p_valor"]).reset_index(drop=True)

    if csv_path is not None:
        df.to_csv(csv_path, index=False)
        print(f"Statistical-tests table saved to: {csv_path}")

    return df


# if __name__ == "__main__":
#     # Example usage: change embedding_type to "amplitude" to test
#     # the other data-encoding type, and dim_reduction to "pca" or "umap"
#     # to choose the reduction method and final number of features,
#     # include_svm_baseline=False if the classical SVM baseline is not desired, and
#     # n_repeats>1 to repeat the experiment with different train/test
#     # splits each time (e.g., n_repeats=10 for a more robust result,
#     # with mean ± standard deviation for each metric).
#     #
#     # IMPORTANT: rerunning this script after changing n_final_features,
#     # n_layers, n_epochs, or dim_reduction does NOT overwrite the previous pickle —
#     # each parameter combination becomes its own key inside the same
#     # <caminho_pickle> (ver gerar_chave_configuracao/mesclar_com_existente).
#     # Example: run once with n_layers=2, then with n_layers=4 — the pickle
#     # will contain BOTH configurations, each with its own
#     # results for each ansatz/SVM.
#     #
#     # In addition, rerunning the script with the SAME configuration does not
#     # retrain anything: the configuration is found in the pickle and the
#     # entire training procedure is SKIPPED (the saved result is returned). Use
#     # force_rerun=True to force retraining anyway.
#     iris_results = run_pipeline(
#         dataset_name="iris",
#         embedding_type="angle",   # "angle" (phase) ou "amplitude"
#         reducao_dim="pca",        # None, "pca" ou "umap"
#         n_final_features=4,       # desired number of features/qubits after reduction
#         n_layers=2,
#         reupload=True,            # True = data re-uploading in all layers
#         n_epochs=60,
#         lr=0.1,
#         batch_size=None,          # None = full-batch; ou defina, ex. 16, 32
#         n_qubits_max=4,
#         n_repeats=1,              # >1 repeats the experiment with different splits
#         incluir_svm_baseline=True,
#         svm_kernels=("linear", "rbf", "poly", "sigmoid"),
#         svm_cv_splits=5,
#         incluir_decision_tree=True,
#         incluir_mlp=True,
#         include_random_forest=True,   # suggested additional baseline (see Section 6)
#         pickle_path="/mnt/user-data/outputs/resultados_vqc.pkl",
#     )

#     # table containing the best configuration of each type (VQC, SVM, Decision Tree,
#     # MLP, Random Forest) for the dataset above
#     table = generate_comparison_table(
#         iris_results, sorting_metric="f1_weighted",
#         csv_path="/mnt/user-data/outputs/tabela_comparativa.csv",
#     )
#     print("\n", table.to_string(index=False))

    # generate_statistical_tests_table requires n_repeats > 1 (the example above
    # uses n_repeats=1 for a quick run) — run again with n_repeats=10,
    # for example, and then:
    #
    # statistical_tests_table = generate_statistical_tests_table(
    #     iris_results, metric="f1_weighted", test_name="wilcoxon", alpha=0.05,
    #     csv_path="/mnt/user-data/outputs/tabela_testes_estatisticos.csv",
    # )
    # print(statistical_tests_table.to_string(index=False))

    # Example usage for the mental-health dataset (uncomment and adjust the
    # CSV path to run). With n_repeats=10, each main disorder is
    # trained 10 times, each time with a different train/test split
    # (the same split is used by all models in each repetition). Then,
    # generate_comparison_table(mental_health_results) produces one row per
    # MODEL TYPE (VQC, SVM, Decision Tree, MLP, Random Forest) PER
    # MAIN DISORDER:
    #
    # mental_health_results = run_mental_health_pipeline(
    #     caminho_csv="dataset_final_mental_health.csv",
    #     embedding_type="angle",
    #     reducao_dim="pca",
    #     n_features_final=8,
    #     n_qubits_max=8,
    #     n_layers=2,
    #     reupload=True,
    #     n_epochs=60,
    #     n_repeats=10,
    #     incluir_svm_baseline=True,
    #     incluir_decision_tree=True,
    #     incluir_mlp=True,
    #     incluir_random_forest=True,
    #     pasta_saida="/mnt/user-data/outputs",
    # )
    # mental_health_table = generate_comparison_table(
    #     mental_health_results,
    #     csv_path="/mnt/user-data/outputs/tabela_comparativa_saude_mental.csv",
    # )
    # print(mental_health_table.to_string(index=False))
    #
    # mental_health_statistical_tests = generate_statistical_tests_table(
    #     mental_health_results, metric="f1_weighted", test_name="wilcoxon",
    #     csv_path="/mnt/user-data/outputs/tabela_testes_saude_mental.csv",
    # )
    # print(mental_health_statistical_tests.to_string(index=False))

In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="pca",
    n_final_features=6,
    max_qubits=6,
    n_layers=15,
    reupload=True,
    n_epochs=500,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=True,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_PREPROCESSING_OK_vs2_1_VERIFICANDO_MORE_LAYERS.csv",
)
print(mental_health_table.to_string(index=False))



In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="umap",
    n_final_features=5,
    max_qubits=5,
    n_layers=10,
    reupload=True,
    n_epochs=300,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=False,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_PREPROCESSING_OK_vs2_2.csv",
)
print(mental_health_table.to_string(index=False))



In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="pca",
    n_final_features=6,
    max_qubits=6,
    n_layers=10,
    reupload=True,
    n_epochs=300,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=False,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_PREPROCESSING_OK_vs2_3.csv",
)
print(mental_health_table.to_string(index=False))



In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="umap",
    n_final_features=6,
    max_qubits=6,
    n_layers=10,
    reupload=True,
    n_epochs=300,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=False,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_vs2_4.csv",
)
print(mental_health_table.to_string(index=False))



In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="umap",
    n_final_features=7,
    max_qubits=7,
    n_layers=10,
    reupload=True,
    n_epochs=300,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=False,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_vs2_5.csv",
)
print(mental_health_table.to_string(index=False))



In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="pca",
    n_final_features=7,
    max_qubits=7,
    n_layers=10,
    reupload=True,
    n_epochs=300,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=False,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_vs2_6.csv",
)
print(mental_health_table.to_string(index=False))



In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="umap",
    n_final_features=4,
    max_qubits=4,
    n_layers=10,
    reupload=True,
    n_epochs=300,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=False,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_vs2_7.csv",
)
print(mental_health_table.to_string(index=False))



In [ ]:
mental_health_results = run_mental_health_pipeline(
    csv_path="../dataset/dataset_final_mental_health.csv",
    embedding_type="angle",
    dim_reduction="pca",
    n_final_features=4,
    max_qubits=4,
    n_layers=10,
    reupload=True,
    n_epochs=300,
    n_repeats=10,
    include_svm_baseline=False,
    include_decision_tree=False,
    include_mlp=False,
    include_random_forest=False,
    output_dir="outputs",
    measure_all_qubits=False,
)

mental_health_table = generate_comparison_table(
    mental_health_results,
    csv_path="outputs_ONE_MEASUREMENT/tabela_comparativa_saude_mental_ONE_MEASUREMENT_vs2_8.csv",
)
print(mental_health_table.to_string(index=False))

